# Argmax: reducing (value, index) pairs

Greedy decoding picks the token with the largest logit: an **argmax** over the vocabulary. On a GPU, argmax is a reduction whose elements are pairs $(v, i)$, combined by

$$\text{best}\big((v_1, i_1), (v_2, i_2)\big) = \begin{cases} (v_1, i_1) & v_1 > v_2 \text{ or } (v_1 = v_2 \text{ and } i_1 < i_2) \\ (v_2, i_2) & \text{otherwise} \end{cases}$$

The tie rule (smaller index wins) makes `best` associative and commutative, so the reduction order does not change the answer. It matches `np.argmax`, which returns the first occurrence.

Write `argmax(x)` with two kernels:

1. **Stage 1**, `BLOCKS = 8` blocks of `TPB = 64` threads: each thread folds its grid-stride elements into a local pair, the block tree-reduces 64 pairs held in **two** shared arrays (`float32` values, `int32` indices), and thread 0 writes the block's pair to `part_val[blockIdx.x]`, `part_idx[blockIdx.x]`.
2. **Stage 2**, one block of 64 threads: reduce the 8 partial pairs the same way and write the winning index to a 1-element `int32` device array.

## Inputs and output
- `x`: `float32`, shape `(n,)`, `1 <= n <= 4096`. No NaNs; `-inf` may appear.
- Return the index as a Python `int`, equal to `np.argmax(x)`.
- Threads with no elements start from the pair $(-\infty, n)$: it loses to every real element, including a real $-\infty$ (whose index is smaller than $n$).

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import math
import numpy as np
from numba import cuda, float32, int32

BLOCKS, TPB = 8, 64

def argmax(x):
    """x: float32 (n,), n >= 1. Returns the index of the first maximum as a Python int."""
    raise NotImplementedError

@cuda.jit(device=True)
def better(v1, i1, v2, i2):
    """True if pair (v1, i1) beats (v2, i2)."""
    return v1 > v2   # TODO: tie rule

@cuda.jit
def stage1(x, part_val, part_idx):
    pass

@cuda.jit
def stage2(part_val, part_idx, out):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [1.0, 3.0, 2.0, 3.0], "dtype": "float32"}], "e": 1}, {"a": [{"__nd__": [-4.0, -1.0, -7.0], "dtype": "float32"}], "e": 1}, {"a": [{"__nd__": [2.5], "dtype": "float32"}], "e": 0}, {"a": [{"__nd__": [{"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}], "dtype": "float32"}], "e": 0}, {"a": [{"__nd__": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], "dtype": "float32"}], "e": 0}, {"a": [{"__nd__": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "dtype": "float32"}], "e": 700}, {"a": [{"__nd__": [0.3983699679374695, -0.5628233551979065, 0.5888349413871765, 0.04211810231208801, -1.570900559425354, 1.0016547441482544, -0.09787619113922119, 0.6198021769523621, 1.8368321657180786, 0.2684299647808075, -1.074468731880188, -0.6809789538383484, 1.0168262720108032, -1.456768274307251, 0.2635892629623413, 0.3752644956111908, -1.1503044366836548, 0.16926391422748566, 0.5562222003936768, -0.7727903723716736, -0.48048901557922363, 0.5437745451927185, -1.2988967895507812, -0.880154550075531, 0.9538697004318237, 0.08562352508306503, -1.4161607027053833, 0.01137579046189785, 1.9412344694137573, -1.513156533241272, 0.8159111142158508, 0.7587356567382812, -0.17678484320640564, 0.6052823662757874, -0.6741008758544922, 1.5447455644607544, 1.7201638221740723, -0.8800106644630432, -0.21365387737751007, 0.1318831890821457, -0.6253682971000671, 0.3527009189128876, -1.0375882387161255, 0.5773811340332031, -0.3006659746170044, -0.048793114721775055, 0.4711216688156128, 0.9262511730194092, -0.9836953282356262, 0.7893146872520447, 0.8790770769119263, -0.11516803503036499, 1.5377178192138672, -0.7645990252494812, -0.5454217791557312, 0.38205811381340027, -0.6803091168403625, 1.3926975727081299, 0.19801437854766846, 1.719524621963501, -1.0541456937789917, -0.9494599103927612, 1.3357129096984863, -1.048083782196045, 1.0756192207336426, -0.8072109818458557, 0.26504939794540405, 1.1657336950302124, -0.05924307182431221, -1.1466573476791382, -0.4870750904083252, 1.1385127305984497, 0.19461823999881744, 0.7087190747261047, 0.08802235871553421, 0.28938591480255127, -0.18700231611728668, -1.012582540512085, 0.54709792137146, 0.1863364428281784, -1.9782158136367798, 0.00960727408528328, 0.9758106470108032, 1.9359549283981323, -1.3514527082443237, 0.11670363694429398, 0.5613536238670349, 0.761428952217102, -0.6939648389816284, 0.5148183703422546, -0.4136577248573303, -0.04173083230853081, -0.5295783877372742, 0.31144294142723083, -0.38553425669670105, 0.30602404475212097, -0.08113007992506027, 0.9197116494178772, 1.6859692335128784, -0.0749104917049408, -0.7302666902542114, -0.2267349660396576, -0.5302294492721558, -1.655869722366333, 0.028118275105953217, -0.4875994026660919, 0.9048224091529846, -0.12548233568668365, 0.9299336671829224, 0.7705214619636536, -0.4951073229312897, -0.32647213339805603, -0.5232079029083252, -0.4520390033721924, 0.8219935894012451, -0.5183369517326355, -0.4197579026222229, -0.805558443069458, 1.3980598449707031, -0.3627890646457672, -0.15844233334064484, -0.8703616261482239, 0.22345155477523804, -0.7927225232124329, -0.6793032288551331, -2.0495243072509766, -1.658007264137268, 0.17048728466033936, 0.7399483323097229, -1.6830437183380127, 0.1424054205417633, 0.16520148515701294, 3.0218722820281982, -1.0140968561172485, -0.6921901106834412, -1.0733435153961182, 0.7994938492774963, 1.0208820104599, -0.29373809695243835, 1.4924548864364624, 2.078599452972412, -0.4878225326538086, 0.5294150710105896, -0.21404767036437988, -0.653978168964386, -0.4823295772075653, -0.25421813130378723, -1.3015562295913696, -0.10104065388441086, 0.7841373085975647, -0.9550715088844299, -0.5346168279647827, -0.9207459688186646, 0.3683609068393707, -0.11396010220050812, 0.9199299812316895, -0.3240019977092743, -1.4774104356765747, 0.5555979013442993, -0.3067752718925476, 1.1344558000564575, 0.7093860507011414, -2.634882688522339, -1.9811365604400635, 0.022280512377619743, -0.7609233856201172, -2.2506425380706787, -0.5439951419830322, 1.2800382375717163, 0.6257850527763367, -0.8953576683998108, 0.2767138183116913, 1.0045394897460938, -0.19380316138267517, 1.7637417316436768, -0.7175432443618774, 2.096034049987793, 1.5256187915802002, 0.5116987824440002, 0.21799787878990173, 0.02581978030502796, -0.9410243630409241, 0.9739081263542175, 1.266541838645935, 0.7784620523452759, 0.2758921980857849, 0.10091385245323181, -0.9368841052055359, -0.6608312129974365, -1.4505460262298584, 0.47811657190322876, -0.9074896574020386, 0.8329179286956787, -0.6675740480422974, 1.415845274925232, 1.5632152557373047, 1.527586579322815, 1.1383864879608154, -0.30972790718078613, -0.31900444626808167, -0.7421092987060547, 0.3238275647163391, 1.6710841655731201, -1.095108985900879, 2.4176487922668457, -0.5736087560653687, -1.1739506721496582, 0.3337307572364807, 1.0707870721817017, -0.36400988698005676, -1.0267815589904785, 1.7475208044052124, -1.1062108278274536, 0.9894372820854187, 0.14589989185333252, -0.37892264127731323, 0.04194045066833496, -1.0934569835662842, -1.1177952289581299, 0.30131423473358154, -1.0437290668487549, -0.7755081057548523, 1.4439853429794312, 1.793436050415039, 1.102664589881897, 0.8905781507492065, -0.2566400170326233, 0.08625428378582001, -0.7429275512695312, -2.8125905990600586, 0.19129221141338348, 0.21681848168373108, 0.9818021059036255, -1.2036559581756592, 1.1421085596084595, -0.5458781123161316, -1.2460274696350098, -0.2671200633049011, 0.9188879132270813, -0.02205521985888481, 0.42989569902420044, -0.8930094242095947, 0.4647481441497803, 0.9707857370376587, -0.8974867463111877, 0.40677696466445923, -1.3573155403137207, 0.21479260921478271, -1.5088841915130615, 0.391032338142395, -0.020121300593018532, -1.1352967023849487, 0.34682872891426086, -0.9983789324760437, 0.071988046169281, -0.9252163171768188, 0.5238272547721863, -0.5708334445953369, 1.4205971956253052, 0.44443586468696594, 0.5278156995773315, -1.3379510641098022, -2.0906894207000732, -0.8025450110435486, 1.8044054508209229, -0.9440231919288635, 1.2175440788269043, 1.2526196241378784, -0.5858309268951416, -0.21324460208415985, 1.3048535585403442, -0.25751200318336487, -0.4076448082923889, -0.5768250226974487, 0.07120678573846817, -0.11050641536712646, 1.233142614364624, 0.2362097203731537, 1.1785446405410767, -0.10663138329982758, 2.501824378967285, -0.11551519483327866, -0.14455871284008026, -0.7963451743125916, -1.685807466506958, -0.8641388416290283, 0.21421989798545837, -0.9875335097312927, 0.8298587799072266, -1.6229287385940552, -1.6674343347549438, 0.3073166608810425, 0.38440102338790894, -1.1236897706985474, 1.0832585096359253, 1.386494517326355, 0.36592572927474976, -1.2764629125595093, 1.2490781545639038, -0.2790292203426361, -0.16817434132099152, 0.48966893553733826, -0.6858956217765808, 1.5150400400161743, -0.611127495765686, 1.0721023082733154, 0.7804098725318909, 1.7760918140411377, 0.5145334601402283, 0.872999906539917, -0.3747938871383667, -0.21574287116527557, -0.5855346322059631, 0.7729025483131409, -0.6250790953636169, 1.5301445722579956, -0.08206962794065475, -1.151098370552063, -0.6318567395210266, 1.1888347864151, -0.44553783535957336, 0.6244438290596008, -1.5309876203536987, 0.4877004027366638, 0.8116167783737183, 0.5496281385421753, 0.8302781581878662, -0.9278134703636169, -1.5901073217391968, -1.2470000982284546, 0.8940398097038269, 0.4607327878475189, 0.8621744513511658, -0.4908043146133423, 0.868258535861969, -1.4633278846740723, 0.48042985796928406, -0.9569036960601807, -0.3505765497684479, 0.7309696674346924, 0.9505026936531067, -0.07390445470809937, -0.2673835754394531, 1.4298195838928223, 1.1212315559387207, -0.5792999267578125, 0.38540247082710266, -0.11075691878795624, 1.0614558458328247, 0.556229293346405, 0.5353245735168457, 0.2007661759853363, -0.4235633611679077, 1.2011146545410156, 0.5206940174102783, -2.181941270828247, -0.26237231492996216, 0.3787422478199005, -0.6187946796417236, -0.8004322052001953, -0.515971302986145, 0.4602462947368622, -0.8327856063842773, 0.4755679965019226, 1.4592170715332031, 1.0530056953430176, 1.0899707078933716, -0.09703204780817032, 1.2888758182525635, 0.135300874710083, 0.990628719329834, -1.0078366994857788, 1.3542287349700928, -0.4635664224624634, 0.14398248493671417, 1.1588134765625, 0.11144557595252991, -0.3577311336994171, -0.05314096063375473, 1.1917599439620972, 1.205700397491455, 0.7627493143081665, -0.5229163765907288, -0.30365216732025146, 0.7489184737205505, 0.8347371220588684, -0.6146876811981201, -0.6034114360809326, 0.7802126407623291, 1.1707963943481445, 0.16983669996261597, -1.093273639678955, 1.6631609201431274, 0.12260069698095322, 0.543872594833374, -2.0543935298919678, -0.6428513526916504, -0.11888106912374496, 0.37301021814346313, -0.8230317831039429, -1.4733806848526, -1.5883128643035889, -0.3443187177181244, 0.31965765357017517, 0.2740575075149536, -1.6900205612182617, -0.6869695782661438, -0.08668103814125061, 0.9909611940383911, 0.5545075535774231, 0.257411390542984, -1.662959098815918, 1.072648286819458, -0.09264756739139557, 0.20771972835063934, -0.6707233190536499, -0.20925530791282654, -0.3441035747528076, -0.4120218753814697, -0.8842287659645081, 0.6564927101135254, 0.8823878169059753, -0.5424115657806396, -0.8323497772216797, -0.8219718933105469, -0.311558336019516, 1.3925421237945557, -0.14591535925865173, 1.7904887199401855, -1.2590402364730835, -1.08748459815979, -0.7860252857208252, -0.8615106344223022, 0.3094601333141327, 1.704405665397644, -0.7354980707168579, 0.3276314437389374, 1.2997368574142456, 2.3604278564453125, 0.20135965943336487, 0.9673500061035156, 1.3055886030197144, 0.09656606614589691, 0.5624687075614929, -0.34293898940086365, -0.8683929443359375, -0.578453540802002, 1.1453286409378052, 1.3305054903030396, 0.25663474202156067, -1.1930389404296875, 1.2281363010406494, 0.014403424225747585, 1.4936751127243042, 0.11825034022331238, -1.116174578666687, -0.25916802883148193, -0.6358856558799744, 1.5898407697677612, -2.2172482013702393, -0.559334933757782, -0.2847575843334198, -1.9895967245101929, 0.44577252864837646, 0.6328825950622559, 0.8638368248939514, 0.8484571576118469, -0.0688844546675682, 0.9558798670768738, -0.41226768493652344, 0.057790230959653854, 0.2450292706489563, 0.09989342093467712, 0.5335529446601868, -0.6191850304603577, -0.09372350573539734, -0.5524297952651978, 0.0944051593542099, 0.33456724882125854, -0.7140495777130127, 1.4557238817214966, 1.9432966709136963, 0.581628143787384, 0.12757040560245514, -0.5804004669189453, -0.13168367743492126, -0.1758112609386444, 0.7212424874305725, 0.0332808718085289, 0.7351557612419128, 0.5997048020362854, -0.43204599618911743, -1.2639877796173096, -1.1177797317504883, -1.0044902563095093, 0.019135214388370514, 1.2099082469940186, -0.8636919856071472, 0.6886874437332153, -0.12357117980718613, 1.2928212881088257, 2.264603614807129, 1.1012378931045532, 0.38828256726264954, 0.5553745627403259, 1.3514925241470337, -0.22425858676433563, -0.7018705606460571, 0.23411300778388977, 1.651904821395874, -0.6278104186058044, 1.1671473979949951, 0.9333824515342712, -0.14972023665905, -0.711591362953186, -1.54385244846344, -1.079544186592102, 0.22466547787189484, 0.7854856848716736, -0.5647091865539551, -0.5103344321250916, -0.7114615440368652, 2.45745587348938, -1.3123345375061035, -1.4176017045974731, 1.0878146886825562, -1.0484082698822021, 1.0214637517929077, -0.7922706604003906, 1.0434328317642212, 1.3887895345687866, -0.43401476740837097, 8.108054316835478e-05, -0.961540162563324, 0.8867675065994263, -0.29611286520957947, -0.9252870082855225, 0.44712701439857483, 0.37589001655578613, -0.0379202626645565, -0.42007729411125183, 0.9575637578964233, -0.6442155838012695, -0.8516021370887756, -0.24220988154411316, 0.5164817571640015, -0.3855556547641754, -0.6284083724021912, -1.0197919607162476, -0.7982693314552307, -0.669579029083252, 1.9667329788208008, -1.7825857400894165, -0.46754541993141174, 0.05180932208895683, -0.8918577432632446, -1.1974177360534668, 0.5389589071273804, 1.3590247631072998, 1.0838793516159058, 0.0046361335553228855, -0.45230627059936523, 0.6453648209571838, -1.6528505086898804, -0.848498523235321, 1.4481141567230225, 0.41613903641700745, 0.150600865483284, -0.19732527434825897, -0.39007648825645447, -0.8506547808647156, 0.5670197010040283, 0.49653246998786926, 0.510857880115509, -2.4390926361083984, 1.2722254991531372, 0.8504410982131958, 1.5595757961273193, -0.5651054978370667, -2.173119306564331, -1.304462194442749, -0.8898327946662903, 2.0040338039398193, -0.754679799079895, -0.46709364652633667, 0.9933551549911499, 0.8920281529426575, 0.967444896697998, -0.46658632159233093, 1.8668200969696045, 0.400590181350708, -0.8471441268920898, 0.40823468565940857, -1.4113872051239014, -1.2436569929122925, -0.6824879050254822, -1.186561107635498, -0.496928870677948, -0.651616632938385, -0.47242388129234314, -0.8733118176460266, 1.1165024042129517, -0.9970889687538147, -0.5330908894538879, -0.9118337631225586, -0.594438910484314, -1.296677827835083, 2.104158878326416, -0.27343565225601196, 0.819695234298706, -0.3648037612438202, -1.3014448881149292, -0.5148003697395325, -0.6815332770347595, -1.228591799736023, 0.10822881758213043, 1.6146429777145386, 0.19452142715454102, 1.0430351495742798, 1.3001309633255005, -1.9240658283233643, 1.9097051620483398, -1.3058812618255615, 1.8458397388458252, 0.22801032662391663, 0.6169200539588928, 2.2248661518096924, 1.0695761442184448, -1.5854687690734863, -1.7807824611663818, -1.179413914680481, 1.0243470668792725, 0.33473554253578186, 0.917290210723877, 0.4439936578273773, 1.227432370185852, -1.0252019166946411, -1.8296377658843994, -0.6675164699554443, 0.8135729432106018, -1.010115146636963, -2.3403546810150146, -0.5448373556137085, 0.6834467053413391, 0.4144056737422943, -0.3444041609764099, -0.5153371095657349, -2.113295555114746, -0.04606710374355316, 1.8649771213531494, -0.5292801260948181, 1.5930343866348267, -1.2815643548965454, -0.4647194743156433, -0.5410632491111755, -0.09226369857788086, -1.8747493028640747, 1.098716378211975, -0.423907071352005, -2.2700793743133545, 0.526268720626831, 0.02016202174127102, -1.397334337234497, -0.23833978176116943, 0.6928777694702148, 0.5859319567680359, 0.24093428254127502, -0.7568883299827576, 0.20499157905578613, 1.215551495552063, 2.118302345275879, 0.5859703421592712, 0.7832938432693481, 0.9465112090110779, -0.2716832756996155, -1.7637059688568115, -0.5887899994850159, -0.2324886918067932, 0.16415613889694214, -0.09738518297672272, -0.8703367710113525, -0.7271910309791565, -0.4568210542201996, -1.1310241222381592, -1.1203843355178833, -2.035752058029175, -0.7082801461219788, -0.27250054478645325, -1.054302453994751, -0.689920961856842, -0.3513701856136322, -1.5456753969192505, -1.108830451965332, -1.381771445274353, -0.0160691998898983, -1.5873699188232422, -0.43918296694755554, -2.485260248184204, 1.2840160131454468, 0.10568113625049591, -1.2317883968353271, -0.46395042538642883, 1.506198525428772, -0.2670653462409973, -1.8264588117599487, -0.3632313311100006, -0.7953968048095703, 0.22360055148601532, -0.0918453261256218, -1.8403197526931763, 0.025426629930734634, 0.7783516645431519, 1.2329078912734985, 0.1957189440727234, 1.7284421920776367, 1.3301293849945068, 0.667401134967804, 1.4522489309310913, -2.600910186767578, 1.1693785190582275, 1.373555302619934, -0.17228130996227264, 0.75239497423172, -0.720693051815033, -0.23443390429019928, -0.24424943327903748, -1.1150482892990112, -0.288635790348053, -1.8392506837844849, -0.2140241414308548, 1.6216660737991333, 1.819352626800537, -0.09183075278997421, 1.6710846424102783, 0.341617614030838, -1.3401001691818237, -0.6857172250747681, -0.11872577667236328, -0.07982087880373001, 0.5921055674552917, -0.7612068057060242, -0.4649913012981415, 0.9869654178619385, 0.08944852650165558, 0.371580570936203, -0.09449101984500885, 0.6718555688858032, 0.7299525737762451, -0.7947155833244324, 1.1209548711776733, -0.6583572626113892, 0.3390865623950958, -0.3077692985534668, 0.657490611076355, 0.27709507942199707, 0.5704746246337891, 0.13887372612953186, -0.11460187286138535, 0.20204462110996246, -0.014038460329174995, -0.8305920958518982, -1.5561273097991943, 0.5272437334060669, 1.278658151626587, -0.9671890735626221, 0.5014591813087463, -0.6124541759490967, 0.1781141608953476, 0.8743245601654053, 0.35639488697052, -0.24232979118824005, -1.943281650543213, -0.21704697608947754, -0.03984074294567108, 1.21573007106781, -0.7095065712928772, -0.12006015330553055, 0.17402172088623047, 1.2340835332870483, -1.2041535377502441, 0.8028140664100647, -0.11926668137311935, -0.6905680298805237, -1.3833256959915161, -0.9851556420326233, 0.8009748458862305, -0.5388734936714172, -0.9661772847175598, -0.4256938397884369, 1.300697922706604, 0.9629649519920349, 0.797764241695404, 0.9554122090339661, 0.7549993395805359, -0.4232042729854584, -1.109417200088501, 1.2123349905014038, -0.1365026831626892, 0.11210380494594574, -0.47543632984161377, 0.49124717712402344, 1.3260750770568848, -0.4135217070579529, 0.1432044506072998, 0.05500909313559532, 0.652570903301239, -0.013616524636745453, 0.039026983082294464, -0.593517541885376, 0.0367438979446888, -0.714841365814209, 0.734650194644928, -0.12794530391693115, 1.638982892036438, -0.6679742336273193, -1.0144506692886353, 0.9125953912734985, 0.63531494140625, -1.353283405303955, 2.589370012283325, -0.16294261813163757, 1.1144274473190308, -0.3830244541168213, 0.6908880472183228, 1.1072461605072021, -0.47045063972473145, 0.35940876603126526, 0.8769183158874512, 2.046729326248169, 0.9967979192733765, 1.4155349731445312, 0.6117845177650452, 0.7268278002738953, 0.5987740159034729, -0.1630275398492813, -1.7762579917907715, -1.632193922996521, 0.38013172149658203, 0.511320948600769, 0.09446662664413452, -0.5525303483009338, -0.47956332564353943, -1.243844747543335, 1.173581600189209, -1.6536084413528442, 1.6202956438064575, -0.43605735898017883, 2.5184664726257324, 0.47730308771133423, -0.3216090500354767, 0.11180570721626282, -0.4901909828186035, 1.4935858249664307, -0.04224288836121559, 0.2153361290693283, -0.8937457203865051, 0.5532225370407104, -1.5097324848175049, -0.6954763531684875, 1.2098562717437744, -2.0931193828582764, 0.25675955414772034, 1.132045030593872, 0.08030633628368378, 1.420248031616211, 1.2077577114105225, 0.4505312442779541, -1.623508334159851, -0.5122811198234558, -0.3725181519985199, 0.9432956576347351, -0.14390742778778076, -0.3536328375339508, -1.3493516445159912, -0.2791873514652252, 0.1391148716211319, -2.518967390060425, -0.16297240555286407, 0.26834970712661743, -0.37681981921195984, -0.2623310685157776, -0.6840070486068726, 0.7632519006729126, -0.6717082262039185, 1.054174780845642, 0.22793453931808472, -2.020160436630249, -0.42497289180755615, 0.15241014957427979, 0.7058115601539612, -0.011135243810713291, -0.2583596110343933, 0.6761887669563293, 1.4499366283416748, -1.909873366355896, -0.748225212097168, 1.5462700128555298, 0.16084957122802734, -1.0907371044158936, 1.5790224075317383, 1.8384231328964233, -0.33625248074531555, 0.637650728225708, -0.08813726156949997, 0.25633466243743896, 0.8663948774337769, -0.3663460910320282, -0.17856454849243164, -0.4097348749637604, -1.8103142976760864, 0.3450862467288971, 0.7139117121696472, 0.047247014939785004, -0.07098502665758133, 0.690178394317627, -0.8106505870819092, -1.161849856376648, 0.3347167372703552, 0.03564583137631416, 1.615074872970581, -1.5256538391113281, -0.1616654396057129, -1.3330246210098267, 1.9917001724243164, -0.5203002095222473, 0.6474099159240723, -2.3145201206207275, -1.0648078918457031, 0.6481710076332092, -0.5146395564079285, -0.7172321677207947, 0.5662063956260681, 0.010894504375755787, 0.021459495648741722, 0.15983732044696808, -0.5789586305618286, 1.4230564832687378, 0.9138372540473938, -0.2250821590423584, -0.4537714719772339, -1.9202346801757812, -1.2062244415283203, 1.220033049583435, -1.2599021196365356, -0.5005389451980591, 1.5624902248382568, 0.9734141826629639, 0.2570517957210541, 0.32006388902664185, -0.3992142081260681, 0.09604157507419586, -0.03540728613734245, 1.2329093217849731, 0.6847703456878662, -0.7927257418632507, 1.3324809074401855, 0.7782779335975647, 1.052881121635437, 1.2003456354141235, 0.7384994626045227, 0.17535901069641113, 1.2679327726364136, 1.1325339078903198, -0.24520033597946167, -0.1647692322731018, 0.2582760453224182, 0.5168308615684509, -0.039864957332611084, -1.3610543012619019, 1.1722862720489502, -0.6563359498977661, -1.1349684000015259, -0.8341087698936462, 0.04871567338705063, -0.23485374450683594, -2.289867401123047, -1.5571646690368652, -1.6025632619857788, -0.79230135679245, 0.0741281732916832, -0.6965914964675903, 1.8726454973220825, 0.804627537727356, 1.3858671188354492, 0.6741184592247009, -1.2696220874786377, 0.4443798363208771, -1.4674233198165894, -1.0009032487869263, 0.20603002607822418, 0.03499787673354149, 0.6132118701934814, -0.24531574547290802, 1.0733989477157593, 0.8154224157333374, 0.3738451898097992, -0.6789408326148987, -0.541747510433197, 1.0790011882781982, 2.874537944793701, -0.4002322554588318, 0.234723299741745, -0.79757159948349, -2.3480324745178223, 0.6651502251625061, 0.22090186178684235, 0.732513427734375, -0.5638056397438049, -0.7616100907325745, 1.5139354467391968, -0.8602277040481567, 0.7349490523338318, 0.9853045344352722, 0.03700976446270943, -0.5148734450340271, 0.27373284101486206, -1.700979471206665, -0.24651910364627838, -0.3384750485420227, -1.361134648323059, -0.2583233714103699, 0.8996103405952454, 2.299562931060791, -1.0511914491653442, 0.46168252825737, 0.4477677345275879, -0.25659462809562683, -0.5244441032409668, 0.5262634754180908, 0.7564428448677063, -1.365601658821106, 1.5253816843032837, -0.10053840279579163, 0.14620047807693481, -0.5219208598136902, 1.2479280233383179, -1.2689629793167114, -0.43663227558135986, 0.16868053376674652, -0.8713908791542053, -0.2106185108423233, 0.5248094201087952, 1.1776262521743774, -0.15140379965305328, -0.43554389476776123, 0.48172953724861145, 0.13395963609218597, -0.9927249550819397, 0.8684621453285217, -0.2589845061302185, -0.8557233214378357, 0.9692235589027405, 0.5815402865409851, -0.7431989908218384, 0.6891195178031921, -0.6614527702331543, -0.5305048823356628, -0.9632962346076965, 0.8436795473098755, -0.12786589562892914, 0.5856210589408875, -1.2735776901245117, -0.4221903085708618, -1.6594594717025757, -0.654874861240387, -0.421273797750473, -0.22405250370502472, -1.5458523035049438, 0.9914492964744568, -1.6529357433319092, 1.1167470216751099, -0.17533063888549805, 0.9823474287986755, -1.0530034303665161, 0.06442102789878845, 0.5163692831993103, 2.097806930541992, 0.381289541721344, 1.714355230331421, 0.3473735749721527, 1.408431053161621, 0.04168606176972389, 0.15464046597480774, 1.0063104629516602, -1.1326903104782104, -1.4805612564086914, 0.29366952180862427, 2.218433380126953, 1.5685627460479736, 0.2372022420167923, 0.7607132792472839, 0.372416615486145, 0.5519625544548035, 1.5691304206848145, -0.9551841616630554, -0.9973424673080444, -1.7302906513214111, -0.6867534518241882, 1.1826093196868896, 0.10278589278459549, -1.1140981912612915, -0.46115121245384216, -0.3036900758743286, 1.1678452491760254, 0.6389156579971313, 0.5583019256591797, 1.376173973083496, 0.7062846422195435, 1.2627202272415161, -0.5929834246635437, -0.7431366443634033, -0.17955365777015686, 2.1888654232025146, -0.9353806972503662, 0.3621366620063782, 0.6946587562561035, -0.7631046175956726, -1.0425010919570923, 2.187162399291992, 0.016916340216994286, -0.20560775697231293, 0.5131139755249023, 0.2544327676296234, -1.2791693210601807, -1.5811607837677002, -0.07829422503709793, -0.6591677665710449, 0.047547824680805206, -1.0718950033187866, 2.628603219985962, 0.5564910173416138, 2.085327386856079, -0.3599918782711029, 0.028921306133270264, -0.3374488353729248, 0.2263304442167282, 0.5998436808586121, -0.3307468295097351, -0.8743451833724976, -0.5224606990814209, -0.046953488141298294, 0.9539437890052795, -1.409775972366333, -0.6599765419960022, 0.48675498366355896, 1.4046939611434937, -0.3649471402168274, 0.35905247926712036, -0.14275908470153809, -0.2028791606426239, 0.18178518116474152, -0.4964430034160614, -0.21051272749900818, -0.28072455525398254, 1.5522209405899048, -0.001782748382538557, 1.640971064567566, -0.10695333033800125, 0.15626488626003265, -0.8157776594161987, 0.18399551510810852, 1.3698148727416992, 1.5699816942214966, -1.0627204179763794, 2.4117982387542725, 0.3095320463180542, 0.500511884689331, 0.9675764441490173, 0.04540075734257698, -0.8438016772270203, 0.04619209095835686, -0.35855770111083984, 1.888166069984436, 0.03811228275299072, -2.2099969387054443, 1.2995091676712036, 0.5794159770011902, -0.0515008419752121, -1.8477914333343506, 3.198331832885742, -0.34542545676231384, -0.3498174548149109, -0.5795313715934753, 0.14863154292106628, -0.09705672413110733, 0.29682379961013794, 0.4604502022266388, 0.003738009836524725, 1.0778169631958008, -1.0306758880615234, 0.9144034385681152, 0.36015748977661133, 0.023482754826545715, 0.7135415077209473, 0.2556859254837036, 0.2197515219449997, -0.44568824768066406, -0.8260077238082886, -1.4525600671768188, -1.229737401008606, -0.6363754868507385, 0.7831506729125977, -0.23910507559776306, 0.11870001256465912, -1.0597617626190186, -1.5230563879013062, 0.05508078634738922, 2.252180814743042, -1.1277343034744263, 1.2594305276870728, -1.1864826679229736, -1.2355647087097168, -0.9693008065223694, -0.9099217057228088, 0.7274554371833801, 0.09804598987102509, -0.01079113595187664, -0.6175419092178345, 0.7956671118736267, 1.6795504093170166, 0.19237710535526276, -0.4459756314754486, -1.1066930294036865, -0.39228999614715576, -1.2922701835632324, 1.2408289909362793, -0.801880419254303, 1.0799767971038818, -1.2317543029785156, 0.8113524913787842, 0.5503286719322205, 0.7272712588310242, 0.28399229049682617, -1.0336878299713135, 2.6446571350097656, -1.3797731399536133, 1.0804427862167358, -0.18877433240413666, -1.083207607269287, 0.7658942341804504, 1.3948173522949219, 0.9891179203987122, -1.318053960800171, -0.7500712871551514, 0.5335259437561035, 0.36342382431030273, 0.8834103345870972, -0.24178603291511536, -1.7789297103881836, -1.2670950889587402, 0.1792171150445938, -1.110313892364502, -1.4585615396499634, 0.4833937883377075, 1.2283766269683838, -0.7528086304664612, -0.3065112233161926, -0.2324620634317398, 0.7852238416671753, -0.3673574924468994, -0.37019744515419006, -0.1626722812652588, -1.3131951093673706, 1.1718330383300781, 1.4024789333343506, -0.04928044229745865, 0.025024838745594025, 0.11582167446613312, -0.0618676021695137, -0.821678638458252, 0.5689805746078491, 0.621947705745697, 0.9170273542404175, -1.054940938949585, 0.08750803023576736, -1.379724383354187, 1.6969680786132812, -1.5866296291351318, -0.23766738176345825, 1.4259727001190186, -1.799899935722351, 0.01415132824331522, -1.7470086812973022, 0.7941550612449646, -0.46810442209243774, -0.6846054792404175, -1.0310931205749512, 0.40637335181236267, 0.02366029843688011, 0.5874778032302856, 0.7138519287109375, -0.6493499875068665, 0.1884721964597702, 1.206438660621643, 0.44817474484443665, 0.3582710325717926, 0.07991935312747955, -0.4576769769191742, 0.008045974187552929, -0.06027434393763542, 2.420367956161499, -0.27125319838523865, -0.9667348861694336, 0.3711101710796356, -0.21881532669067383, -0.6099127531051636, 0.8025814890861511, -0.1691018044948578, -0.7848275303840637, 0.5718247294425964, 1.927436351776123, 1.300612211227417, 0.2629601061344147, 0.31290403008461, -0.01874241791665554, 0.7117524743080139, -0.18578536808490753, 0.3637707233428955, -0.14548367261886597, 0.21515889465808868, -1.9475080966949463, -0.007208697497844696, -0.5445341467857361, -0.3203314542770386, 0.36977115273475647, -1.2461074590682983, -0.8635180592536926, 0.1597370058298111, 0.04805975407361984, 0.8861178159713745, 0.32268455624580383, -1.5890103578567505, 0.3872522711753845, -0.16526418924331665, 0.12012708187103271, 0.9697868227958679, 0.0905754491686821, -0.20945821702480316, -1.5856410264968872, -0.4442884922027588, 0.7856146097183228, 0.5027088522911072, -1.2097805738449097, -1.368610143661499, -0.39817190170288086, 0.4274412989616394, -0.7445821166038513, 1.3071513175964355, 2.065782070159912, -1.1205881834030151, 1.9710286855697632, 1.8835525512695312, -2.023293972015381, -1.643189549446106, -0.19113700091838837, -0.03780386224389076, -0.884243369102478, -0.04568647965788841, 0.7241255044937134, -0.9209244847297668, -2.2657480239868164, -0.3668612837791443, 0.7229662537574768, 1.3951913118362427, 0.9944947361946106, -2.7786855697631836, -1.2340452671051025, -0.1677733063697815, -0.9032631516456604, 0.4808531701564789, 0.3890608847141266, 0.8220043778419495, 1.3714404106140137, -0.5718240737915039, 1.7143913507461548, -1.4036952257156372, -1.0639902353286743, -1.09709632396698, -0.8866723775863647, 0.6495926380157471, 1.655333161354065, -1.4631942510604858, 0.7334018349647522, 1.2051751613616943, 0.2989063858985901, -1.9766250848770142, 0.47136199474334717, 0.9235367774963379, -1.733923316001892, -0.4499691426753998, 1.0964018106460571, 0.3934938907623291, 1.1862019300460815, 1.3318710327148438, -1.1789225339889526, -1.1188149452209473, 0.9608791470527649, -0.2582891285419464, 0.9188104867935181, 0.6206403970718384, -0.757607102394104, 0.9752370715141296, 0.4821331799030304, -0.5965930223464966, -0.49006280303001404, -0.5587224364280701, 0.28949490189552307, 0.9261183738708496, 0.5072033405303955, -0.2095087319612503, -1.4158746004104614, 0.041151005774736404, -0.16147471964359283, -0.7236256003379822, -0.9584314227104187, 1.468930721282959, 1.4083598852157593, -1.4825074672698975, 0.8687736988067627, 0.2814873456954956, 1.6718335151672363, -0.9290058016777039, 0.8488219976425171, -0.6819263696670532, -0.36568087339401245, 0.06727703660726547, 0.7933658957481384, -0.021923378109931946, -1.6923409700393677, 0.6667299270629883, -0.16961294412612915, -0.3946487605571747, -1.8748533725738525, -0.5987252593040466, 0.9793357253074646, 1.071318507194519, 0.29510533809661865, -1.04751718044281, 0.5972923636436462, -1.017462134361267, 0.615193784236908, -0.699381947517395, -1.6824185848236084, 0.26677829027175903, 0.037197601050138474, 0.037425193935632706, -1.0528888702392578, -1.2019412517547607, 0.9426836967468262, -0.21433845162391663, 1.4546635150909424, 1.4576983451843262, -1.2348761558532715, -0.5288646817207336, -3.0560004711151123, -0.2536182999610901, -0.8365170955657959, 0.043171484023332596, -0.5259636640548706, -0.7184537053108215, -0.9974852800369263, -0.9522467851638794, 1.0224579572677612, 0.45596641302108765, 1.956964373588562, 0.6146907210350037, -0.5599883794784546, -0.6414865255355835, 1.441370964050293, -2.0341057777404785, -0.059849683195352554, -0.32488223910331726, -2.2325053215026855, 0.933845043182373, -0.12250230461359024, -0.3612242639064789, -0.9953334927558899, -0.19082868099212646, -0.2717142701148987, 0.8674283027648926, -0.4661765694618225, 2.280686140060425, -0.7239094376564026, 0.022659294307231903, -0.7553362846374512, 0.7301046848297119, -0.5286396145820618, 1.3278635740280151, 0.9159687757492065, 0.32923465967178345, -1.742091417312622, -0.10740949213504791, -0.17985276877880096, 0.5200645923614502, -1.5007704496383667, 0.5314607620239258, 0.5746792554855347, -0.1836715191602707, 0.17553967237472534, -0.6033970713615417, 0.8260489702224731, -1.2963236570358276, -0.791899561882019, 2.4276530742645264, -0.18117283284664154, -0.15521353483200073, -1.0319466590881348, -0.361027330160141, 0.16093923151493073, -1.267648696899414, -1.5330040454864502, -0.5016777515411377, 1.5073192119598389, -0.37264135479927063, 2.51743221282959, 1.1513139009475708, 0.5884920358657837, -0.5161628723144531, 0.47105294466018677, 2.9125311374664307, -1.3654197454452515, 0.355185866355896, 1.6533282995224, 0.953450620174408, 0.09914939105510712, -0.46304789185523987, 0.9040687084197998, -1.7724156379699707, 1.7643588781356812, -0.8760436177253723, -2.035876750946045, -0.36266136169433594, -0.2185017168521881, -0.428993284702301, -1.7139254808425903, -0.8375192880630493, 0.6682149767875671, 0.2770341634750366, 1.0643478631973267, 0.08040269464254379, -1.1596359014511108, -0.381233811378479, 0.0003884384350385517, 0.15882210433483124, 0.24652794003486633, 0.20791220664978027, 0.63682621717453, 0.6779758334159851, -0.8514913320541382, 1.1036267280578613, 0.6215579509735107, -0.39362967014312744, 0.1558891385793686, -0.25419196486473083, 0.16052308678627014, -1.9085782766342163, -1.9423208236694336, 0.7742939591407776, -0.4220258593559265, 1.303688406944275, 1.4504655599594116, 0.2892943024635315, 1.8615518808364868, -0.934011697769165, 0.21141459047794342, -1.2517989873886108, 0.2101270705461502, 1.1577802896499634, 0.21446317434310913, 0.3537989556789398, -0.9434249997138977, 0.29605233669281006, -0.7344227433204651, 0.14493714272975922, -0.6870735287666321, 0.9540877342224121, 0.6426795125007629, 0.4465228021144867, 0.8647603988647461, 1.3203593492507935, 0.060918886214494705, -0.6495351791381836, -0.5631181597709656, 1.6962597370147705, 0.40950828790664673, -1.0637662410736084, -1.0098198652267456, 1.230210781097412, -0.22206321358680725, 1.749027967453003, 1.5395197868347168, -0.23682473599910736, -0.2837359309196472, -0.9274701476097107, 0.7485384345054626, -1.404070258140564, 0.39799582958221436, -0.42710426449775696, 1.456902027130127, -0.4241587817668915, -0.8485195636749268, 0.5063769817352295, -1.11350679397583, 0.5732895135879517, -0.4223955273628235, -0.7151168584823608, -0.45400169491767883, -0.3848532438278198, -1.3843798637390137, 0.4436489939689636, -0.20843054354190826, -0.7467121481895447, 1.1591325998306274, 0.1133333295583725, 1.5123666524887085, -1.8882948160171509, 1.2509015798568726, 0.05806485936045647, 1.6046688556671143, 0.029918350279331207, -0.5299575328826904, -0.6751998066902161, 0.3509160876274109, -1.6083251237869263, -0.4155937433242798, -1.0564396381378174, 2.2633447647094727, 0.45493197441101074, -0.33355382084846497, -3.0160629749298096, -0.20443245768547058, -0.7217216491699219, -0.1835090070962906, -0.11119352281093597, -0.8628454804420471, -1.4352530241012573, -0.4450693726539612, -0.279506653547287, -0.20927318930625916, -0.1798267662525177, 0.2679688036441803, -0.9747573733329773, 0.7342084646224976, -0.5324636697769165, 0.03850546479225159, -0.5816774964332581, 0.13574597239494324, 1.4527034759521484, 0.3727911710739136, 0.0506487600505352, 0.017440984025597572, -0.681556224822998, -1.3336907625198364, -2.1919162273406982, -0.015580246224999428, 0.3767477869987488, 0.5503000617027283, -2.0331523418426514, -0.959372878074646, 0.26233983039855957, -0.591280996799469, 0.5586156845092773, 0.7682584524154663, -1.428275465965271, 0.7052812576293945, -0.2436063587665558, -0.6506639122962952, 0.1682499200105667, -0.795167088508606, -1.7851746082305908, 0.7111555337905884, 0.41948094964027405, 0.4111654460430145, 0.05104837939143181, -1.5230295658111572, -0.13090677559375763, 1.5233573913574219, 0.28489309549331665, -0.06655093282461166, 0.1483631730079651, -0.31269028782844543, -0.05888677388429642, -0.091932512819767, -0.7743054032325745, -0.31834927201271057, -0.1804543435573578, -0.10067352652549744, 2.315497875213623, -0.5037367939949036, -1.3887139558792114, -0.293567419052124, -1.9024626016616821, 1.6351059675216675, -0.7664352059364319, -0.32650259137153625, 0.037296079099178314, 0.7583040595054626, -0.7039137482643127, 0.21618129312992096, -0.34112274646759033, -0.1595584750175476, 0.4691501259803772, -0.7759618163108826, 1.3457552194595337, -0.2302199751138687, 0.7801067233085632, -0.6640450954437256, -0.11990527808666229, -0.8839995265007019, 0.6577085256576538, -0.12055722624063492, -0.7650203108787537, -1.1742653846740723, 2.0981435775756836, 0.3959980905056, 0.6614568829536438, 1.2032431364059448, 0.4955660402774811, 1.319567322731018, -0.07563313841819763, 0.33007216453552246, 1.763944387435913, -1.1313289403915405, 0.22076672315597534, -0.568795382976532, 1.6116784811019897, -0.2669796943664551, 0.2127533257007599, 0.13057346642017365, -0.8446853160858154, -0.32992270588874817, -0.6565447449684143, 0.9292783737182617, -0.6639821529388428, 1.1471211910247803, 0.3583545982837677, 0.8555348515510559, 1.3797327280044556, 0.9688726663589478, 0.5000815987586975, -0.5921441912651062, -0.09923512488603592, 0.17365677654743195, 0.83470219373703, 0.08933969587087631, -0.15740883350372314, 1.9473711252212524, 0.5460131764411926, 0.6102950572967529, -0.6239453554153442, -0.4656227231025696, -0.4206152856349945, -0.9344132542610168, 0.022263452410697937, 0.290301650762558, -0.7132341265678406, -0.49901121854782104, 1.0011775493621826, -1.5769444704055786, -0.15881210565567017, 1.4774278402328491, 0.8616620302200317, 0.6972886919975281, 1.7576836347579956, 1.4154291152954102, 0.7953510284423828, -0.5855507850646973, -1.2514747381210327, -0.7972143888473511, 1.2708721160888672, -0.6517088413238525, -0.4900299608707428, 1.0691068172454834, -0.32841557264328003, 0.3424500823020935, -1.3969141244888306, -0.6161454319953918, -0.66265469789505, -0.7682579755783081, -1.6601792573928833, -0.8980701565742493, 0.783494770526886, 0.0714670866727829, 0.7213690280914307, -0.7853558659553528, 1.776036262512207, -0.6730751991271973, -0.41564151644706726, 0.3793363571166992, -0.505629301071167, 0.1499609649181366, 1.4948972463607788, 0.5088397860527039, -0.4035380482673645, 1.2564717531204224, -0.5391898155212402, -0.3880740702152252, 0.3412279784679413, 0.4145181179046631, -2.0191681385040283, 0.4020184874534607, -0.47722819447517395, -1.2548072338104248, -1.2621523141860962, -1.199487566947937, -1.6694482564926147, 0.7847365140914917, 1.2577630281448364, -0.22098951041698456, -0.10375263541936874, -0.06004041060805321, 1.0901356935501099, 0.3628525137901306, 1.7075659036636353, 0.4783998727798462, 1.426665186882019, -1.5336999893188477, -0.9825811386108398, 0.13048937916755676, -0.23069757223129272, -0.8438035845756531, -1.1319173574447632, -1.5249706506729126, -0.016676008701324463, -0.40090787410736084, 0.7329358458518982, -0.5810626149177551, 0.03730425983667374, 0.5429509878158569, 0.8125211000442505, -0.49183526635169983, 0.011330449022352695, 1.8167383670806885, -2.2653119564056396, 0.3685532510280609, 0.3598520755767822, -0.2502915859222412, -1.7685819864273071, -0.039143480360507965, -0.6810629367828369, -0.47825342416763306, -0.8232138156890869, -1.1236982345581055, -0.05755394697189331, -1.3493800163269043, 0.7707046270370483, 0.10124514997005463, -0.4994487464427948, 0.06070515513420105, 0.8565335869789124, -0.2919620871543884, 0.8896862268447876, 2.658726215362549, 0.2726250886917114, 0.5910236835479736, -0.43631553649902344, 0.15205109119415283, -0.6309980154037476, 1.653005838394165, -0.6446682214736938, 0.6417540907859802, -0.610736072063446, 0.10442144423723221, -0.9314300417900085, -1.0702418088912964, 1.2973783016204834, 1.0057694911956787, 0.177269846200943, -0.39612284302711487, -0.20422251522541046, -1.1999948024749756, 0.09829988330602646, 1.3414878845214844, -0.8855040669441223, 2.2958984375, 0.1825711876153946, 1.492478370666504, 0.4883390963077545, -1.176856279373169, -1.3780192136764526, 1.3809105157852173, 1.136085867881775, -1.0178004503250122, 2.0162301063537598, -0.2304231822490692, 3.2644333839416504, -0.5120616555213928, 0.9058716297149658, -0.022499967366456985, -0.8949586749076843, 0.585067629814148, -1.0180370807647705, -1.2282689809799194, -0.3350330889225006, -1.7509315013885498, -1.6616599559783936, -0.6036847233772278, 0.0003101869369857013, 0.6930394172668457, 1.155287265777588, -0.29603415727615356, 1.567014217376709, 0.22206495702266693, -0.27992722392082214, 0.9570993185043335, 1.2443654537200928, -0.08205483108758926, -0.5521637201309204, 0.31807565689086914, -0.14809277653694153, -1.482018232345581, -0.14999230206012726, -0.18099212646484375, 0.020138738676905632, 0.31053972244262695, 1.181448221206665, -0.9977792501449585, 0.06400143355131149, 0.7196373343467712, -2.5287225246429443, 1.2251752614974976, 0.14555127918720245, 2.7755331993103027, 0.7947665452957153, -0.9388832449913025, 0.07208270579576492, -0.32515183091163635, 0.42267298698425293, -1.5572528839111328, 0.2224487066268921, -0.335543692111969, -0.3221798241138458, 0.4307754337787628, 0.1133294552564621, 1.9916877746582031, 0.28825265169143677, 0.7328888773918152, -0.6631004810333252, -0.7090762257575989, 1.5438014268875122, -0.5974363088607788, -1.2625982761383057, -1.2857539653778076, -0.933941125869751, -1.189224362373352, -0.8622274398803711, 1.2119901180267334, -0.2817266583442688, 0.4927363991737366, 2.073025941848755, -0.7264302372932434, -0.6798808574676514, -1.788554072380066, 1.0616122484207153, 0.28999418020248413, -1.8990099430084229, 1.2395853996276855, 0.9483482241630554, -2.8509480953216553, -1.3316776752471924, 0.7203547954559326, -1.1991357803344727, -0.4221554696559906, 0.4751909673213959, 0.17531169950962067, -0.7029789090156555, 0.5609084367752075, 1.2746870517730713, -0.4061673879623413, 2.7242825031280518, 0.6540088057518005, 0.4330142140388489, -1.2497403621673584, -1.0036901235580444, -0.8813075423240662, 0.9336841106414795, -1.0463175773620605, 1.3164563179016113, 0.9948015809059143, -0.628517210483551, -0.08044271916151047, 1.5934224128723145, -0.8235486745834351, -2.2396321296691895, -0.670818567276001, 0.2202414721250534, 1.0416910648345947, 0.6008499264717102, 2.681373357772827, 1.215288758277893, 0.48748141527175903, 1.3750659227371216, 1.4282147884368896, 0.38883355259895325, 0.9979926347732544, 0.8099668622016907, 0.3777691721916199, 0.1386391967535019, -0.986200749874115, -0.022238658741116524, -0.8428937196731567, 0.8270038366317749, 0.17539942264556885, -1.1199754476547241, -0.09228421747684479, -0.28846997022628784, -1.4347734451293945, -0.6573283672332764, 0.13896191120147705, 1.5454133749008179, 0.013578759506344795, 0.5687726140022278, 1.4247866868972778, 1.182375431060791, 0.2028913050889969, 0.6870409846305847, -0.19361227750778198, -0.3607337176799774, -0.007503425236791372, 0.14830118417739868, 0.9949468970298767, 0.6776071786880493, 0.9438514113426208, -1.6675559282302856, -0.41825443506240845, -0.15381290018558502, -1.2923791408538818, -2.4325623512268066, 0.378886878490448, 2.0763182640075684, -1.1054325103759766, 0.08076638728380203, 1.8173480033874512, -0.19314882159233093, -0.2770892381668091, -1.2118494510650635, -0.1854855865240097, 0.8205381631851196, 0.9691545963287354, -0.30985385179519653, 0.9408075213432312, -1.2955204248428345, -0.5900778770446777, 0.21204353868961334, 0.21869930624961853, 0.9748069047927856, -0.28544139862060547, -2.200316905975342, 1.145647644996643, 1.9874306917190552, 0.4421861469745636, -0.32304510474205017, -0.5793992877006531, -0.00203347229398787, -1.5449938774108887, 0.005150750279426575, -0.5147400498390198, -1.7210800647735596, -0.04698535054922104, 0.33425191044807434, 1.6425833702087402, -0.1338188350200653, 1.4769834280014038, -0.7183576822280884, -0.5571753978729248, 0.2385186403989792, -0.9045577645301819, 0.0032152938656508923, -0.9717382788658142, -1.0135780572891235, -0.6626856327056885, 0.5353323221206665, -0.1923787146806717, -0.6791990399360657, 1.4445462226867676, -0.21893733739852905, 0.629359781742096, 0.23393651843070984, 0.7491430640220642, -0.10401102155447006, -0.7037177085876465, 0.2961342930793762, 0.3916291892528534, 0.6180209517478943, 0.503281831741333, 0.45721858739852905, -0.9800747632980347, 1.0320559740066528, 1.0327723026275635, -0.03193613141775131, 0.3611016273498535, 0.24117934703826904, 1.550940752029419, -1.3724273443222046, -0.08889027684926987, 0.8227130770683289, -1.3696857690811157, -0.40532463788986206, 1.927963137626648, -1.067017674446106, 0.1773080676794052, -0.6141557097434998, 0.08914007991552353, -0.545080304145813, 0.3637583255767822, -1.2300207614898682, 1.8921873569488525, 0.6140255928039551, 0.5946748852729797, -0.16427955031394958, 0.4804017245769501, 0.05507121607661247, 0.8242945075035095, -0.07187411189079285, -0.06947007775306702, 1.6428676843643188, 1.3492399454116821, 1.9448163509368896, 1.2426692247390747, -1.3622596263885498, 0.8091374635696411, 1.1552890539169312, 0.6170437335968018, 1.3819873332977295, 1.3863310813903809, 0.664991557598114, 0.8881818652153015, 0.37096649408340454, 1.7376917600631714, -0.8978980183601379, -0.9950958490371704, 1.3396583795547485, -1.2437092065811157, 0.8150748610496521, 0.46333491802215576, 1.623624324798584, -0.5480276346206665, -0.5861615538597107, 0.25218406319618225, 0.21715964376926422, -0.4318051338195801, -0.560987114906311, 0.3270757496356964, 1.0862044095993042, -1.7678619623184204, 1.224980354309082, 0.7996453046798706, -0.1518312245607376, -0.6969797611236572, -0.3689154088497162, -1.7131445407867432, 0.3048112392425537, 0.2170010358095169, 0.15588834881782532, 0.5232558846473694, -1.0170044898986816, 1.0604194402694702, 0.02600514516234398, 0.25644993782043457, 0.46339377760887146, 0.4638897180557251, 0.9598476886749268, 0.41512542963027954, 0.40004196763038635, -0.637029230594635, -0.3179965019226074, -1.3202540874481201, 0.3578018546104431, -0.14770080149173737, 0.053085047751665115, 0.7198947668075562, -0.2285122275352478, 1.4609856605529785, -1.03744375705719, 1.5146225690841675, 1.1900806427001953, -0.2162868082523346, 0.5014931559562683, 1.0493731498718262, -0.4382655918598175, 0.36616575717926025, 1.0529899597167969, 0.5887706875801086, -0.23895055055618286, -1.3362449407577515, 0.762057900428772, 0.8480835556983948, 1.6917304992675781, 0.9130868911743164, -1.1342569589614868, -0.9432931542396545, -1.3890926837921143, 0.6295483708381653, -0.20045873522758484, 0.1216336041688919, -0.5683119893074036, -1.6692253351211548, 0.1447998285293579, -1.772647738456726, -2.1072797775268555, -1.1196407079696655, 0.25123488903045654, 1.1561448574066162, 0.21169790625572205, 0.0452217161655426, 0.3835691809654236, 0.7867440581321716, 1.1306674480438232, -0.8942294716835022, -0.6785204410552979, 2.917320728302002, -1.3624720573425293, -2.267935037612915, -0.6724734902381897, -0.6134432554244995, -0.8992252945899963, 1.584244966506958, 1.0057384967803955, -0.9009444713592529, 2.4274182319641113, -0.3935578465461731, 0.2160876840353012, -0.7634603977203369, -0.8662052154541016, 0.524054229259491, 1.5607277154922485, 0.6975454092025757, 0.1366942971944809, -0.13620077073574066, 1.4453647136688232, -1.3341566324234009, -0.1673654317855835, -0.8316776752471924, -0.15095119178295135, -0.22166380286216736, -0.6788010001182556, -1.0631498098373413, 0.42780059576034546, 2.2171072959899902, 0.4544166326522827, -0.029757242649793625, -0.6850636601448059, 0.7317812442779541, 0.5548343658447266, 2.1287035942077637, -0.1376660019159317, 1.7231154441833496, -0.6765922904014587, 0.6311913728713989, 0.9351686835289001, -0.3251466155052185, -1.5791256427764893, 0.357500821352005, 0.5241561532020569, 1.4123635292053223, 1.2923613786697388, 1.972831130027771, -0.7205797433853149, 1.794244408607483, 0.6560320854187012, 1.0019820928573608, -1.7749577760696411, 0.6495758295059204, 0.2594192326068878, 0.3083711862564087, 2.7949466705322266, -0.48705777525901794, -0.21974186599254608, 0.4085135757923126, 0.47882768511772156, -0.9580605626106262, -1.030531406402588, 0.09677743166685104, -1.0877948999404907, 0.7066994309425354, -0.4062187373638153, -1.3257057666778564, 0.7595365643501282, 0.5965909957885742, 0.7136005163192749, 0.2255198061466217, 0.6631720662117004, -0.24157042801380157, -1.607513427734375, -0.25148141384124756, -0.18790750205516815, -0.35475921630859375, 0.34384435415267944, 1.7659505605697632, 0.9134359359741211, 0.579799234867096, 0.740286111831665, -0.8504812121391296, 2.08528470993042, 0.28836607933044434, 1.42842435836792, 1.1934022903442383, 1.1964614391326904, 1.8779640197753906, 0.17600105702877045, 1.1227378845214844, 0.0017087154556065798, 0.14714783430099487, -1.9325634241104126, 0.2430601418018341, 0.05821837857365608, 0.4417465627193451, -0.7673472166061401, -1.005763292312622, 0.4157279431819916, -0.10859262198209763, 0.5962212681770325, -0.9187625646591187, -1.4248363971710205, 0.5434942841529846, 0.37391600012779236, 0.3028928339481354, 0.4857180714607239, 0.2914617955684662, -0.04165750369429588, -0.7990025877952576, 0.27068665623664856, -0.02675425447523594, -0.29654359817504883, -1.3482697010040283, -0.4145266115665436, 0.549382209777832, 0.7183834910392761, -1.361056923866272, 0.9639250636100769, -0.549463152885437, 1.0509196519851685, -0.11250662803649902, -0.9435857534408569, 1.0469907522201538, -1.3660986423492432, 0.40811190009117126, 0.040493372827768326, 0.5181527733802795, 0.6608129739761353, 0.39695754647254944, 1.2619105577468872, -1.466979742050171, 0.8566921353340149, 1.3707197904586792, -1.332831621170044, -1.3187605142593384, -0.4967135488986969, -0.9970738887786865, 1.0120946168899536, 0.6443253755569458, -0.5871596932411194, -0.4174264669418335, 1.7474541664123535, -0.4527074992656708, -0.6059198975563049, 2.0056140422821045, 1.343200445175171, -0.2585538923740387, 0.7551743388175964, -1.4817672967910767, 1.0737571716308594, 2.99902606010437, -0.290872186422348, 0.8268711566925049, 0.021971143782138824, -0.7695387601852417, 0.005841588601469994, -0.3669661283493042, -1.1754943132400513, 0.17043264210224152, 0.4553017020225525, 0.6988725662231445, 0.5442287921905518, 0.48258543014526367, 0.2659563422203064, -0.20420798659324646, -1.220910906791687, 0.15129145979881287, -0.298760324716568, -1.2544000148773193, 0.07082261145114899, 0.7357687950134277, 1.5874347686767578, -0.7429469227790833, 1.342059850692749, -0.5754639506340027, 0.31937292218208313, 1.377215027809143, 0.11114553362131119, -0.2387559413909912, -0.8956375122070312, 1.1655499935150146, -0.7525460720062256, 0.2735610902309418, -0.0007785034831613302, 0.5036873817443848, 1.480143666267395, 0.6029500961303711, 2.1405954360961914, 1.833310604095459, -0.02492186799645424, -1.1353144645690918, -0.1811303049325943, -0.23427850008010864, 1.450919508934021, 0.06315441429615021, 0.6603268980979919, -1.4095714092254639, 0.6201876997947693, 1.6573396921157837, 0.15840253233909607, -0.7753614783287048, -0.13236254453659058, 0.23947997391223907, 0.15751193463802338, 0.11586012691259384, 0.38849931955337524, -0.7271758317947388, -0.17138321697711945, -0.27977582812309265, -0.2683568000793457, -1.7313430309295654, 0.11747469007968903, 0.38766565918922424, -1.2491360902786255, 1.5163370370864868, -0.276228666305542, -0.16937176883220673, 0.4372265636920929, -0.007893798872828484, 0.8758545517921448, 0.9797733426094055, -2.201165199279785, -0.08937863260507584, 0.8328551650047302, 0.4933525025844574, 0.748483419418335, 0.14010833203792572, -0.3448992073535919, -1.246127963066101, 1.6844648122787476, 0.30941617488861084, 0.21170450747013092, 0.31902366876602173, -0.04606335237622261, 0.1591273546218872, 1.1514756679534912, 0.45975634455680847, -1.2001917362213135, -0.04945434257388115, -0.39850369095802307, 0.008450551889836788, -0.14847545325756073, 1.3841321468353271, -1.1905173063278198, 0.5905276536941528, 0.4996922016143799, -0.5770044922828674, -1.513501524925232, -0.04330476000905037, -1.5640738010406494, 1.163351058959961, 0.5878629684448242, -2.3628337383270264, 0.04681054875254631, -0.47835204005241394, -0.8767505288124084, -0.9403737187385559, 1.4544097185134888, -0.9648343324661255, -0.822812557220459, 0.9503502249717712, -0.18066798150539398, -1.7178958654403687, 2.699939727783203, 0.2819196283817291, -0.937234103679657, 2.623612403869629, 0.40595653653144836, 0.2515959143638611, -0.3599604368209839, 0.7329040765762329, 0.419856458902359, 0.24366624653339386, 1.2660948038101196, -0.0878060907125473, -0.8269368410110474, -1.1114020347595215, 0.7561895251274109, 0.3630836308002472, -0.439963698387146, 1.078442931175232, 0.6526255011558533, -0.6109281182289124, -1.4061155319213867, -0.5564692616462708, -1.1309374570846558, -0.17025898396968842, 0.2548595070838928, 0.46151891350746155, -2.2057583332061768, -0.575562596321106, 0.31842318177223206, 0.4292284846305847, 0.1409020870923996, -0.48764148354530334, -0.012071129865944386, 0.948605477809906, -0.33721205592155457, -0.7377962470054626, 0.5730946063995361, 0.7226059436798096, -1.1206858158111572, -0.6691488027572632, -1.406747817993164, -0.17385978996753693, 0.3248113691806793, -0.2724599540233612, -0.9587647318840027, -1.0021345615386963, 0.0013693092623725533, 0.8287330865859985, 0.16358524560928345, 0.042139165103435516, 1.3258618116378784, -0.24728451669216156, -0.28499945998191833, -0.9534415602684021, 1.298609972000122, 0.9983232617378235, -0.06064987555146217, 0.24532558023929596, -1.0672037601470947, -0.7137245535850525, 0.13601310551166534, 0.6065687537193298, -2.3909895420074463, 1.3856927156448364, 0.6982291340827942, -1.1435891389846802, 0.2298831194639206, -0.49190008640289307, -0.6974673271179199, -0.5752482414245605, -0.587838351726532, 0.05254095420241356, 0.1422879695892334, 0.9166634678840637, -0.21259170770645142, -0.7951730489730835, 0.1284092664718628, 0.4488462209701538, 0.06738674640655518, 1.1918290853500366, -0.31387579441070557, -1.297810435295105, 0.5422431826591492, 0.6061475276947021, -0.0454382486641407, 0.03810605779290199, -1.065748929977417, -0.20259226858615875, 0.15125036239624023, -0.20521433651447296, -1.3682533502578735, 0.05428163707256317, -0.8894092440605164, 0.26267552375793457, 0.09152936190366745, 0.5702641010284424, -0.9089979529380798, 0.12027405202388763, 0.29278942942619324, 0.12860193848609924, 1.0380823612213135, -1.0450242757797241, 0.7243803143501282, 0.9212512969970703, -0.02204781025648117, -1.0161415338516235, -0.12836454808712006, -0.00987839326262474, 0.09229972213506699, 0.9839953184127808, 0.5411733984947205, -1.5000786781311035, 0.8915877342224121, -1.7734241485595703, -0.7601155638694763, -1.7422839403152466, 0.29785165190696716, -0.5697346329689026, 0.9565392136573792, -0.7780736088752747, 0.0650501474738121, 1.3827134370803833, -0.35881367325782776, 1.0971636772155762, 0.24708221852779388, 0.035911668092012405, -1.3535009622573853, -0.053922683000564575, 1.6389461755752563, -0.08565418422222137, 0.41004613041877747, 1.3280582427978516, -0.643178403377533, 0.05321255326271057, 0.07434689998626709, -0.6078028678894043, 0.5068541169166565, 1.2191537618637085, 0.5111913681030273, -0.04680792614817619, 0.15484127402305603, -1.623008370399475, 0.38557472825050354, -0.9913970232009888, 0.9482090473175049, -0.8146352767944336, 0.889897882938385, -0.6042442321777344, -0.5516493916511536, -1.6435346603393555, 0.2662399411201477, 0.23626674711704254, 0.9232085943222046, -0.8761208653450012, -1.6463661193847656, 0.7430304288864136, -2.612703323364258, -1.1437381505966187, 0.7361595034599304, -0.25660061836242676, -1.4093576669692993, -0.3786219656467438, 1.1708868741989136, -0.27415505051612854, -0.48112961649894714, 0.9700000882148743, -0.6893258094787598, 0.04591963067650795, 1.1271120309829712, -1.6937373876571655, -0.8249631524085999, 0.6136082410812378, -0.18074308335781097, 1.6224395036697388, -0.2025141716003418, -0.008205452933907509, -1.4691293239593506, -1.152382254600525, 1.349038004875183, -0.8048256039619446, 0.11409164220094681, -0.40497562289237976, 2.1547093391418457, -1.709116816520691, 0.4162406027317047, -0.1291683167219162, -1.1372127532958984, -0.42211073637008667, 0.4788055419921875, 2.6467292308807373, 1.186951994895935, 0.5174925327301025, 0.13300387561321259, -1.4496088027954102, -1.4161967039108276, 0.4463329613208771, -1.5308057069778442, 1.050451636314392, 0.491975337266922, 1.3616445064544678, 0.31131696701049805, -0.8704343438148499, 1.1700929403305054, -0.9795746803283691, 0.03502028435468674, -0.17459139227867126, -0.7811018824577332, -0.1257762908935547, 0.08061463385820389, -0.8266128301620483, 1.4563170671463013, -0.4798032343387604, -0.38025176525115967, 0.4914192855358124, -0.3036405146121979, -0.08523771166801453, 0.14910988509655, -1.6989704370498657, 1.7739826440811157, -1.0325881242752075, -1.7131702899932861, 0.29994845390319824, 0.3464726209640503, 0.14934460818767548, -0.8853653073310852, -0.9498013854026794, -0.017034655436873436, -0.14751246571540833, 0.9185080528259277, -0.10617940872907639, 0.4734937846660614, -0.11704975366592407, -1.8027944564819336, -0.9455909132957458, 1.3785066604614258, 0.20983384549617767, 1.1820331811904907, -0.03548485040664673, -0.05998693406581879, -0.5109338760375977, 1.0818228721618652, 1.514333963394165, -0.8535469770431519, 0.2885301113128662, -0.6955157518386841, 0.1554701328277588, -1.4138885736465454, -1.324050784111023, 0.5587069392204285, 0.23740732669830322, -0.062432557344436646, 1.4357391595840454, -0.6918472051620483, -0.1394897699356079, 1.2319157123565674, 0.010694773867726326, -0.5214647650718689, 0.9555028080940247, -0.14222826063632965, -0.35247737169265747, -1.2054387331008911, -1.4922380447387695, -1.631066918373108, -1.0055040121078491, -0.35460859537124634, 0.2798401117324829, -1.450624942779541, 0.1634238362312317, 1.2909300327301025, 1.3482041358947754, -0.5681163668632507, 2.5908865928649902, 0.7418533563613892, 1.0367811918258667, 1.6804786920547485, -0.9794968366622925, -0.4068955183029175, 0.0325460247695446, 0.3500160574913025, -1.0604957342147827, 0.8925407528877258, -0.552620530128479, 0.33439549803733826, 0.683826744556427, -0.6105512380599976, 1.4067823886871338, -0.7971600890159607, -2.0406689643859863, 1.1057658195495605, -0.07992052286863327, -1.583032250404358, -0.3359522223472595, 0.33720701932907104, -0.14715659618377686, 0.6522866487503052, -0.9053734540939331, 0.005958848632872105, -0.45988690853118896, -0.3922794461250305, 0.27882328629493713, 0.9985013604164124, -0.26852214336395264, -0.4473837912082672, 0.642425537109375, 0.17221149802207947, -2.263430595397949, -0.23045289516448975, -0.603846549987793, -1.0936017036437988, -0.742302656173706, -0.8164230585098267, 1.745446801185608, -0.7041175365447998, -0.5518112778663635, -2.0525131225585938, -0.41780000925064087, 0.5660528540611267, -0.8080405592918396, 0.30456748604774475, 0.06904814392328262, -0.2427661418914795, 2.5559558868408203, -0.9679550528526306, 0.08712228387594223, -1.4657984972000122, 0.6936989426612854, -0.8524587154388428, -0.7577191591262817, 0.5225359201431274, -0.8512071371078491, 0.6486749649047852, -1.9931994676589966, 0.6028401255607605, -0.9457623958587646, -2.213060140609741, 0.8168456554412842, 1.4850202798843384, 0.21612121164798737, 1.4693663120269775, 0.6968197822570801, -0.9169216752052307, 1.438197374343872, -0.262463241815567, 0.9308526515960693, 0.2342967689037323, -0.8988277316093445, -0.4986611604690552, -1.230460286140442, 0.34980329871177673, -1.459609866142273, 1.7964696884155273, -0.024636799469590187, -1.6795635223388672, 0.05829988047480583, -1.2436425685882568, 1.1955974102020264, 0.2777974307537079, -1.1781764030456543, 1.1704301834106445, 0.8181625604629517, 0.34692496061325073, -1.2899140119552612, -1.1071819067001343, 2.1984856128692627, -1.839430332183838, 0.17451021075248718, 0.07513522356748581, -1.354176640510559, -1.5804340839385986, -1.7783610820770264, 0.8786416053771973, -1.0827698707580566, 0.7648599147796631, 0.46221473813056946, -0.0948188379406929, -0.4970162808895111, -0.6595857739448547, 0.068626269698143, -0.7157443761825562, -1.452004313468933, -0.46434029936790466, 1.6102076768875122, 1.2224234342575073, -0.6624814867973328, -1.761991024017334, -1.3229457139968872, -1.010081171989441, 0.07348702102899551, -0.20879527926445007, 0.8694173097610474, 1.5649646520614624, -0.26481327414512634, 0.8977927565574646, 0.2812390625476837, -0.9159321784973145, -0.609269917011261, -1.0386892557144165, -2.0193750858306885, 0.7521790266036987, 0.9089690446853638, 0.4900130033493042, -0.3605117201805115, 0.06170906871557236, 0.27206602692604065, 0.05650072172284126, -0.5020613074302673, -0.18268994987010956, -0.12735740840435028, 0.7836503982543945, 0.9474341869354248, -1.2221180200576782, 2.4510984420776367, 0.4401870369911194, -0.7694339156150818, -1.5933188199996948, 0.3227230906486511, 0.9525469541549683, -0.8910611867904663, -0.455401748418808, 1.586046576499939, 0.867657482624054, 0.6256570219993591, 0.5401369333267212, 1.945918321609497, -0.5438220500946045, 0.25541383028030396, -0.5775431394577026, -1.1752777099609375, -1.6174546480178833, -0.6950621604919434, -0.8642566204071045, -0.6844594478607178, 0.4895407259464264, -0.5729526281356812, -0.9543169140815735, -0.5144206285476685, -0.6811224222183228, -0.363671213388443, 1.0913589000701904, -0.5367003083229065, 0.00416003679856658, -0.29640522599220276, -0.7053511738777161, 0.2518555521965027, 0.3276735246181488, -0.09744728356599808, 0.6793102025985718, 0.026127682998776436, 0.8884499669075012, -0.6131394505500793, 0.6665582656860352, 0.4487094581127167, -0.28234246373176575, -0.6167501211166382, -0.8193397521972656, -1.0996280908584595, -0.6263304948806763, -0.5745069980621338, -0.4385567009449005, 1.870940923690796, -0.266611784696579, -1.051520586013794, 1.24112868309021, 0.85174560546875, -0.7281724214553833, 0.6405420899391174, 0.225491002202034, 0.8048651814460754, -0.5120733380317688, 0.07855307310819626, -0.2673764228820801, -0.6504824757575989, -0.09267432242631912, 1.3090375661849976, -0.34902849793434143, -0.7075495719909668, -0.7141607403755188, 0.5084505081176758, -0.24423334002494812, -1.3179775476455688, -1.2970380783081055, -0.48545026779174805, 0.6237539052963257, 1.270872712135315, 0.9283642172813416, -0.20980426669120789, 0.3086729049682617, 0.9511674046516418, -0.5561588406562805, -3.231637716293335, 1.4801857471466064, 1.0659586191177368, 0.42874449491500854, 1.205814003944397, 0.7131536602973938, -0.6677247285842896, -1.2511669397354126, -0.30504170060157776, -1.0373246669769287, -0.5246058702468872, 0.6893160343170166, -1.3760733604431152, -0.1935109794139862, 1.897350788116455, -0.03600197657942772, -1.418912410736084, 0.17844359576702118, -0.3160122036933899, 0.9630613327026367, -0.5615129470825195, 1.4032485485076904, 1.0545289516448975, 1.784267783164978, 0.3857385218143463, 1.4643174409866333, -2.083181142807007, -1.3354464769363403, 0.14854848384857178, -0.1880326122045517, -0.6599497199058533, 0.23268026113510132, 0.23702441155910492, 0.47474557161331177, -1.7538092136383057, 0.6835783123970032, 0.4233349561691284, -0.4562123417854309, -0.8797080516815186, 0.4675285220146179, -0.28543978929519653, -0.37053853273391724, 0.7090861201286316, 1.4167894124984741, -0.224164217710495, -0.6952357888221741, -0.10971484333276749, -0.5854505896568298, -1.5770211219787598, 0.5200842022895813, 0.1453920602798462, -1.1779478788375854, -0.5622091889381409, -0.4073425233364105, -1.13369882106781, 0.783664882183075, 0.15427266061306, -1.3303438425064087, -1.9227988719940186, 1.6926460266113281, -2.3340682983398438, 0.7624312043190002, 0.348823606967926, 0.5380064845085144, -1.1310783624649048, -0.32130080461502075, -0.4246635138988495, 1.54780912399292, 0.1375587284564972, -1.562019944190979, 0.3787996470928192, 0.8577635288238525, 0.9617297053337097, -0.3778206408023834, 0.02178778313100338, -0.7641705870628357, -0.19791963696479797, 0.41615593433380127, 0.5527099370956421, -1.6766126155853271, 0.08579035103321075, 0.42023351788520813, 0.2942277193069458, -2.6250863075256348, -0.37238019704818726, 1.0205727815628052, 1.6124368906021118, 0.013706598430871964, 0.07530241459608078, -1.2441480159759521, -0.43064412474632263, -0.1672561913728714, 1.8924751281738281, -2.3955698013305664, 1.123032808303833, 1.0448212623596191, 1.1365878582000732, -1.0448291301727295, 0.0656144917011261, -0.24356181919574738, 0.3625463545322418, -0.47122305631637573, -0.7580205202102661, -0.767284631729126, -1.1976795196533203, 1.1394237279891968, -1.0417735576629639, -0.883460283279419, 0.8723869919776917, -0.3634149432182312, -0.28391754627227783, 0.8372995853424072, -0.11483533680438995, 0.4242490530014038, -0.20582647621631622, -2.2834110260009766, -0.8818990588188171, 0.1724501997232437, -0.5575107336044312, 0.938123881816864, -0.6230779886245728, -0.4208618402481079, -0.7045203447341919, -0.5325424075126648, 1.7745112180709839, -0.8763050436973572, -1.9419240951538086, -1.0371352434158325, -1.9317383766174316, -0.24915240705013275, -0.20987577736377716, -0.5918437838554382, -0.8869965672492981, -0.6674560904502869, -2.321976661682129, -0.3451605439186096, -0.8023736476898193, 1.4423354864120483, -0.2819991409778595, 0.009454915300011635, -1.012695074081421, -0.2815414071083069, 0.852903425693512, -1.1528849601745605, 0.05389269068837166, -0.11808644235134125, 0.6726157069206238, -0.43081438541412354, -0.08148148655891418, -0.3165613114833832, 1.5653196573257446, 1.0940693616867065, -0.05289516597986221, 0.6566553115844727, -0.8942846655845642, -0.04332008585333824, 0.0018352658953517675, 0.17389647662639618, 1.459395170211792, -0.2583598792552948, -0.40626850724220276, 0.5799186825752258, 0.38345932960510254, 1.4734083414077759, 3.2363626956939697, 0.8632195591926575, -0.09114573150873184, -0.2422875314950943, 1.478234052658081, -0.8511651754379272, -0.22141627967357635, 0.4516543447971344, 0.4491617977619171, -0.4205912947654724, 1.7009423971176147, 1.3519655466079712, 1.4994416236877441, -0.24571636319160461, -0.7411673665046692, 0.6316479444503784, -0.07448960095643997, -0.11913639307022095, -0.17066870629787445, 1.2168470621109009, -0.5668323040008545, 3.1298909187316895, -2.068354606628418, -0.30989399552345276, -0.11691107600927353, -0.5671740770339966, -0.005583518650382757, -0.27368173003196716, -1.4437487125396729, 0.3424340486526489, 2.0218985080718994, -1.3030701875686646, -0.7198190689086914, -1.0824793577194214, 0.09118206053972244, -1.2958372831344604, 0.047651588916778564, 0.15310493111610413, 0.4918976426124573, 1.1248544454574585, -0.7506818771362305, -0.9301064014434814, 0.5770635604858398, 0.9980242848396301, -0.3162531554698944, -1.845658540725708, 0.8941969871520996, 0.8748595714569092, -2.1961240768432617, 0.6327713131904602, -0.3859555423259735, -1.7806669473648071, 1.1534892320632935, 0.480622261762619, 2.1298325061798096, 0.3372650742530823, -0.06844654679298401, 1.0787187814712524, 2.0826632976531982, 1.0309590101242065, -0.017945237457752228, 0.301093727350235, -0.11307321488857269, -0.9525296092033386, -0.32101061940193176, 0.28275299072265625, -0.4087974429130554, 0.7942588329315186, -0.648739218711853, 0.3626641631126404, -1.218179702758789, -1.7351813316345215, 0.9277044534683228, 0.1899515688419342, -0.40979114174842834, 0.7219262719154358, 2.5378942489624023, 0.46511179208755493, 1.2503974437713623, -0.31276369094848633, 0.22311098873615265, 0.01308142114430666, 2.2712817192077637, 1.0704529285430908, -0.9138347506523132, -1.488348126411438, 2.5760862827301025, 0.7742111086845398, 0.23905640840530396, -0.3162465989589691, 0.058818962424993515, -0.6540926694869995, -0.2680848240852356, -1.1077414751052856, 0.008145385421812534, -0.4006972312927246, 1.130321741104126, 0.40522465109825134, -1.6005138158798218, -0.46121522784233093, 1.0970908403396606, -0.024036435410380363, -0.4246099591255188, 0.7755348682403564, 0.44442272186279297, -0.9141739010810852, 0.6315788626670837, 0.08573123812675476, -1.1276062726974487, -0.5157343149185181, 0.29440224170684814, -0.7763682007789612, -0.07216092944145203, 0.46987321972846985, -0.3356059491634369, -0.9281498193740845, -0.8261028528213501, -0.4258743226528168, -1.391898512840271, 1.7611404657363892, -0.3312755227088928, 0.0890849307179451, 0.055860210210084915, 1.1870249509811401, -1.7245807647705078, 0.6186258792877197, -0.275569349527359, -0.5611076354980469, -1.7494256496429443, 0.6699989438056946, 0.6863081455230713, 0.27728551626205444, -0.3022884726524353, -1.2646223306655884, -0.6843801736831665, 0.8665262460708618, 0.12271563708782196, 1.091267704963684, 0.23687593638896942, 1.0059010982513428, 1.3175437450408936, 1.2439076900482178, -1.0949112176895142, 0.7550193071365356, 0.04132690653204918, 1.7968804836273193, 0.3717188239097595, 1.2668163776397705, -0.44218501448631287, 0.6624605059623718, -0.33722472190856934, -0.5892004370689392, 0.3794901371002197, 0.16776517033576965, 0.6446403861045837, -1.5798356533050537, -1.3033984899520874, 1.6744351387023926, -0.5604410767555237, -0.02021832950413227, 0.741422176361084, 1.0963963270187378, 1.1986961364746094, -0.8172502517700195, 0.8289110064506531, 0.32210373878479004, -0.45294395089149475, 0.02624891884624958, 0.7041710019111633, 0.8726816773414612, -0.21093930304050446, 0.9796942472457886, -1.3781497478485107, -0.09203857183456421, -2.0304744243621826, 1.6521620750427246, 0.5883914232254028, -0.5670786499977112, -0.3815234899520874, -0.16821785271167755, -0.057676203548908234, -1.668565034866333, -0.6969508528709412, 0.7576114535331726, 0.5089931488037109, -0.8622947931289673, 0.44928938150405884, 1.2669557332992554, -0.9905456900596619, -1.7175549268722534, -1.6724647283554077, 1.0278403759002686, -0.062056928873062134, -0.38965359330177307, -1.0002856254577637, 1.2298378944396973, 0.026387283578515053, 0.6507364511489868, -0.10364321619272232, -1.5627418756484985, 1.3601583242416382, -0.4306882321834564, 0.8042581677436829, 0.44204577803611755, -0.4875606298446655, 1.3635696172714233, 1.0569030046463013, 0.658299446105957, 1.6845494508743286, -0.26001906394958496, 0.7786602973937988, -0.25410524010658264, -0.143977090716362, 0.8395617604255676, -1.4757786989212036, 0.5495569705963135, -0.9537187218666077, 0.852486789226532, -1.574947714805603, -0.5364622473716736, 1.1298658847808838, 0.15842223167419434, 0.7745903134346008, -0.40450045466423035, 1.530229926109314, 0.05844345688819885, 1.36813223361969, -0.575925886631012, -0.8523501753807068, 0.7690900564193726, -0.7323299050331116, 0.23866984248161316, 1.8293070793151855, -0.3056553602218628, -1.0090749263763428, 0.7604513764381409, -1.0991154909133911, -0.16695331037044525, -0.5069432854652405, -0.26137593388557434, -0.5096752047538757, -2.628734827041626, 0.31327661871910095, 0.9546402096748352, 0.9382798671722412, 0.22117602825164795, 0.4253764748573303, -1.426346778869629, -1.1009321212768555, 1.4271929264068604, -0.6606054902076721, -1.593579649925232, -0.3255557119846344, -0.6556543707847595, 0.18500059843063354, 0.4118954539299011, -1.4431349039077759, 0.034101665019989014, 0.8111264705657959, 0.28072911500930786, -0.6257079839706421, 0.7966216802597046, 0.15451394021511078, -1.410057544708252, -1.4465563297271729, 0.31568780541419983, -1.3718349933624268, -1.1904335021972656, 0.3207879066467285, -0.17184419929981232, -0.9847004413604736, -0.21219345927238464, -0.8419162034988403, 0.8875617384910583, -0.32878610491752625, 1.9414607286453247, 1.471796989440918, 0.9920307397842407, 0.422228068113327, 0.19575412571430206, -0.6337577700614929, 1.4170730113983154, 2.5063974857330322, -0.19032128155231476, -0.021650955080986023, -3.3279504776000977, -0.30990955233573914, -1.0391772985458374, 2.1558048725128174, -1.1664342880249023, 0.10460849851369858, 0.9657462239265442, 0.595064103603363, -1.1938501596450806, 0.28736522793769836, 0.4633007049560547, 0.022130930796265602, -0.08975512534379959, -2.2769556045532227, 0.14641855657100677, -0.8590911626815796, 0.619573712348938, -1.3852704763412476, -1.5914306640625, 0.06328214704990387, -0.31958892941474915, -1.4954155683517456, -0.5938478112220764, -0.7624193429946899, -0.04955826327204704, -0.07472642511129379, -1.5521165132522583, 0.5134050250053406, 0.9832451343536377, -1.9295680522918701, -0.7372059226036072, 0.8916216492652893, -1.615736722946167, -1.2964026927947998, -1.6931034326553345, 1.2150990962982178, 0.5907995700836182, -1.3406128883361816, 0.5332158803939819, 0.6292341351509094, 0.6563360095024109, 0.7910961508750916, 0.8648226261138916, -0.541374683380127, 0.9818065762519836, -0.7123667001724243, 2.0431644916534424, -0.7217212319374084, 0.34598568081855774, 1.6167542934417725, -0.31828469038009644, 0.07023557275533676, 0.05094999074935913, -0.2964754104614258, 0.6852107644081116, 1.277166724205017, -0.7415380477905273, 1.3857066631317139, -1.0965298414230347, -0.415657103061676, -2.1932179927825928, -1.8302960395812988, 0.5839943289756775, -1.3107030391693115, -0.8497252464294434, -2.0738635063171387, 0.72232985496521, -0.9658552408218384, -1.163029432296753, 1.8098456859588623, 0.1088157668709755, 0.11109966039657593, -1.0818583965301514, 1.8541580438613892, -0.35810592770576477, 0.7362152934074402, 0.5817726254463196, -1.2245607376098633, -0.1042313426733017, 0.42850708961486816, 0.7888526320457458, 0.4715104103088379, -0.6802529096603394, 0.3660629689693451, 0.9005745053291321, -1.413743257522583, 0.13817113637924194, 0.29959169030189514, 0.8729997873306274, -0.08043507486581802, 0.2262992411851883, -2.7662787437438965, -1.1635375022888184, -2.046858310699463, 0.06920936703681946, 1.2397373914718628, 0.7432408928871155, -0.535458505153656, 1.5422308444976807, 0.28871944546699524, 0.6065611243247986, -0.147065207362175, 0.5024145841598511, -0.48087483644485474, 0.5024251341819763, 0.4979895055294037, -0.7430259585380554, 1.2392964363098145, 0.3952164947986603, -1.0319955348968506, -2.0314626693725586, -0.40496429800987244, 0.8105270266532898, -0.584765613079071, 1.0595349073410034, 0.2565181851387024, -0.023603912442922592, 0.4502148926258087, 0.427925169467926, -0.13555379211902618, 0.2109481692314148, 0.4194835126399994, -0.1523963063955307, 2.277214765548706, 0.7762807607650757, 1.0778642892837524, 1.1071653366088867, 0.9196959733963013, -0.08760210871696472, -1.080040454864502, -0.4666469097137451, 1.738137125968933, 0.5488792657852173, -1.4263356924057007, 0.019444409757852554, 0.6217939853668213, -0.791547954082489, -0.31475141644477844, -0.04822757467627525, 1.9017854928970337, 0.08294406533241272, -0.027010945603251457, -1.3930953741073608, 1.1187808513641357, 1.0617449283599854, 0.41686564683914185, -0.022033900022506714, 0.7045556306838989, 0.18176127970218658, 0.5069448947906494, 1.134500503540039, 0.07211387157440186, -0.34033405780792236, 0.4697744846343994, -0.9711896777153015, -1.0195941925048828, 1.284153938293457, -1.4845706224441528, -1.6541186571121216, -0.8398464322090149, -0.3987554609775543, -0.6988805532455444, 0.6783946752548218, -0.2163415253162384, -2.5551939010620117, -1.2058300971984863, 0.060045212507247925, -2.710101366043091, 0.6559381484985352, 0.5406661033630371, -0.28009527921676636, -0.14246413111686707, -1.9134483337402344, 0.021461395546793938, 0.3369179964065552, -1.502199411392212, 0.20285376906394958, 1.1500049829483032, 1.7340896129608154, 0.46192583441734314, -0.005938580725342035, 0.1505705565214157, -0.8393339514732361, 0.007124986033886671, 0.38473525643348694, -1.0522304773330688, -0.6558113098144531, 0.11466920375823975, -0.7160537242889404, 1.2122793197631836, -1.2890825271606445, -0.0897042378783226, 0.1359933316707611, 0.6240642666816711, -0.8104947209358215, -1.046637773513794, -0.22386378049850464, -0.355394184589386, -0.41789311170578003, 0.7854702472686768, -0.8239248991012573, 0.918757975101471, 0.2356075942516327, 1.195069432258606, 0.43215397000312805, 0.6159079074859619, 0.05517648905515671, 0.04314272850751877, -1.799578309059143, 1.346448302268982, 1.5486772060394287, -0.45360201597213745, 0.9071862101554871, -0.36819377541542053, 0.7923412322998047, -0.9307506680488586, -0.040062956511974335, 0.4530273675918579, -0.3287499248981476, 0.7820723652839661, -1.0310932397842407, -0.5892578363418579, 1.0727912187576294, -0.8959885239601135, -0.44853878021240234, -1.7327443361282349, -0.9048061370849609, -0.4167064130306244, 1.4776443243026733, -0.3402194678783417, 0.7577319145202637, 0.19704316556453705, -0.42349696159362793, -0.7507284879684448, 1.4923585653305054, -0.4365466833114624, -0.7800208926200867, -1.7226672172546387, -0.09909642487764359, -0.01917131245136261, 0.22622397541999817, -1.1979401111602783, -1.4642901420593262, -1.8073745965957642, -0.6066856384277344, -0.8557429313659668, -0.5930612087249756, 0.4424374997615814, -0.8030524253845215, 0.4969862699508667, -2.309835195541382, 0.8324152231216431, -0.22427786886692047, -0.6725131869316101, -0.24486945569515228, -0.3812732398509979, 0.4530109763145447, -0.31912437081336975, 1.5631154775619507, -1.7735384702682495, -0.6923003196716309, -0.3981302082538605, 0.47158947587013245, -1.8106428384780884, -0.7211615443229675, 1.7568987607955933, -0.09252649545669556, 1.5606298446655273, -1.2300814390182495, 0.010979668237268925, 0.3449149429798126, -0.6808071136474609, -0.46423354744911194, -0.7697376608848572, 0.3071169853210449, 0.7527613639831543, 1.265608310699463, 0.3865833878517151, 0.09058041870594025, -1.3044801950454712, 2.774850368499756, 2.1651864051818848, -1.9413700103759766, 0.743105411529541, 0.23220230638980865, -0.2652283310890198, -0.6095682382583618, 0.669257402420044, 0.8854721784591675, 1.3701764345169067, -0.6894679069519043, -0.2819344699382782, 0.9329189658164978, 0.3387952148914337, -0.07370281964540482, -0.0012530838139355183, 0.7728358507156372, -0.7186793684959412, -1.2066514492034912, 1.106831669807434, 0.5149797201156616, -0.9109185338020325, -0.1538902223110199, 0.3039828836917877, 1.2539781332015991, 0.4906773269176483, 0.22363366186618805, -0.8673391938209534, 0.030767913907766342, -1.2386192083358765, 0.3954278230667114, 2.0978710651397705, -0.7757930755615234, 0.9654185175895691, 1.7724605798721313, -1.770544409751892, 0.7133060097694397, -0.47690215706825256, 0.25061196088790894, -0.3766706585884094, -0.001791032962501049, 0.5562977194786072, -0.17327460646629333, 0.3111158013343811, 0.3535197675228119, 0.31194454431533813, -0.17786580324172974, -0.38807815313339233, -0.5832672715187073, 1.6556158065795898, -1.4379619359970093, 1.8538368940353394, 0.6431394219398499, 1.172566533088684, -0.14869920909404755, 1.231993556022644, -0.034704841673374176, -0.8685631155967712, -0.17404495179653168, -0.9101641774177551, 2.227748394012451, 0.6596111059188843, 0.18296998739242554, 0.048892199993133545, 1.7545795440673828, 1.3644064664840698, 0.6101778149604797, -0.1498258262872696, 0.16581064462661743, 1.229608416557312, -0.743836522102356, 0.2055177390575409, 0.1529901623725891, -0.6699885725975037, 0.9188425540924072, -0.11824879795312881, -0.620384693145752, -0.9101662635803223, 0.0836368203163147, -1.301403522491455, -0.6654459834098816, -0.17605283856391907, -0.6196497082710266, -0.2058548629283905, 0.22203990817070007, -2.077526807785034, 0.36314722895622253, -0.5626067519187927, -0.2928474247455597, 0.4418703019618988, 0.40933844447135925, -0.7410511374473572, -1.175473690032959, -0.33134889602661133, -2.3571584224700928, 0.31845763325691223, -1.3416426181793213, -0.2796296775341034, -0.6684227585792542, 1.0655039548873901, 1.112511396408081, -0.8676664233207703, 0.8801061511039734, -0.41315507888793945, -0.6556072235107422, 0.09172464907169342, -0.9282472133636475, -0.2411241978406906, -1.5231647491455078, -0.7599020600318909, 0.34584781527519226, -0.9811094403266907, 1.2312037944793701, 0.5605489611625671, -0.9845419526100159, 0.21471627056598663, -0.5611350536346436, 0.8582441806793213, 1.0530250072479248, -0.8663603067398071, -1.7289844751358032, 1.3819489479064941, -0.6374602317810059, -0.6701333522796631, 0.09025149792432785, -1.580913782119751, 1.2109014987945557, -0.33041149377822876, -0.9709329605102539, 2.376220703125, 2.4031171798706055, 0.3332119286060333, 0.43910127878189087, 0.8147138357162476, -0.9458009004592896, -3.181117296218872, 0.8196402192115784, -0.4746795892715454, -0.15495650470256805, 0.31708452105522156, 0.6641407012939453, 1.3934403657913208, -0.10180573910474777, -1.0229768753051758, 0.30396565794944763, 0.5580209493637085, 0.43055832386016846, 1.1087007522583008, 0.6802238821983337, 0.6915535926818848, 0.25416579842567444, -1.1296087503433228, 2.252224922180176, -1.163048505783081, -1.5386643409729004, -0.08884783834218979, 1.855506181716919, 0.39288002252578735, 0.9621356129646301, -3.4015276432037354, -0.03231288865208626, -0.42251095175743103, 0.07191198319196701, 0.6113022565841675, 1.1640030145645142, 0.9301685690879822, 0.5200299620628357, 1.315582513809204, -0.8972054123878479, -0.8869909644126892, -0.4731786549091339, -0.36475661396980286, -0.7974238991737366, 0.01402325090020895, 0.7092606425285339, 0.8506375551223755, -0.15478461980819702, -0.3339831233024597, -0.30529457330703735, -0.42984071373939514, 0.2546454071998596, -1.4733502864837646, -0.609959602355957, -0.48122450709342957, 2.8517215251922607, -0.3721083402633667, -1.2116838693618774, 0.10052777826786041, -1.105051875114441, 1.1705117225646973, 3.4631335735321045, 0.19744475185871124, -0.7216806411743164, 0.664556086063385, 0.6733776330947876, 0.8186102509498596, 0.4554312825202942, 1.8029083013534546, 0.7450897097587585, 0.29800304770469666, 0.7932605743408203, -1.396180272102356, -0.6452189087867737, 1.1573143005371094, -0.3819069266319275, -0.5489228367805481, 0.5972992777824402, 1.791428565979004, 0.0762113556265831, -0.6708648204803467, -0.3100753724575043, -0.17624011635780334, 1.0236601829528809, -1.1677805185317993, 0.3833063840866089, 0.8318982720375061, -0.1729675680398941, -0.6946977972984314, -0.2061719000339508, 2.7843778133392334, 0.44916632771492004, 0.9812629222869873, -1.4815033674240112, -0.3299471437931061, -1.077736258506775, 0.22334899008274078, -0.5659211277961731, 0.5545806884765625, -0.2424408495426178, 1.0317301750183105, 0.057686224579811096, -0.13110469281673431, -0.20124897360801697, -0.6774749159812927, 0.012673446908593178, 0.8216439485549927, -1.051133155822754, 0.5319264531135559, 0.26762568950653076, -0.5415156483650208, -0.4978366494178772, 2.148387908935547, 1.3239126205444336, -0.45510926842689514, -0.2664530277252197, -1.7905209064483643, 0.5009050965309143, -1.2687230110168457, 0.16545891761779785, 0.23985736072063446, -1.9404401779174805, 0.9791476130485535, 0.9415494203567505, 0.4804748296737671, 0.4770819842815399, -0.22713929414749146, 0.7551395297050476, 0.19948755204677582, -0.08542133867740631, -0.6604384779930115, -0.4657820165157318, -0.12471254169940948, 1.144325852394104, 0.6384769678115845, -1.2079317569732666, 1.0811656713485718, 1.133548378944397, 0.0737825259566307, 0.8262071013450623, -1.0464403629302979, -0.9736685156822205, -1.0622071027755737, 0.6872912645339966, 0.29053595662117004, -2.206881284713745, -0.165720134973526, -1.7179392576217651, 0.29342249035835266, -0.21995490789413452, 0.719137966632843, -0.18402239680290222, -0.10781530290842056, 1.039311170578003, 1.924033284187317, 0.18967044353485107, 1.4474222660064697, 2.1004228591918945, 0.7562217712402344, -1.0668277740478516, 0.7804429531097412, -0.5039052367210388, -0.1032780185341835, 1.0683287382125854, 0.2963413596153259, -0.5551669001579285, -1.6620246171951294, 0.8076314330101013, -0.1435472071170807, 0.19099394977092743, -1.2772501707077026, -1.193887710571289, 0.5498678088188171, -0.448603093624115, 0.22757592797279358, 0.45043203234672546, 1.3579777479171753, 10.0], "dtype": "float32"}], "e": 3999}, {"a": [{"__nd__": [-0.9830247759819031, -1.6529844999313354, -0.18978816270828247, 0.4028501808643341, -0.8383662104606628, 1.1109082698822021, -0.0036853444762527943, 0.14996474981307983, -0.030219215899705887, 0.03807907551527023, 0.06933053582906723, 0.30145883560180664, -0.30168455839157104, -0.14842291176319122, 0.6911421418190002, 0.22703920304775238, 0.5312122106552124, 0.31719642877578735, -0.3918663561344147, 0.5812421441078186, -0.658154308795929, 0.8799747228622437, -2.1247453689575195, 0.9134162664413452, -0.801945686340332, 0.9696728587150574, -0.30239444971084595, -0.13435523211956024, -0.511593759059906, -0.9920542240142822, 2.0716195106506348, 0.5812164545059204, -0.8804795742034912, -1.636531949043274, 0.4845174252986908, -0.3371376693248749, 0.3799004852771759, 0.507361114025116, 1.9692294597625732, 0.7409453988075256, -1.4647589921951294, 1.1246044635772705, -0.4944128096103668, -0.36323684453964233, 2.8821561336517334, 0.017906032502651215, -0.8074173331260681, -1.3229933977127075, -0.6650747656822205, -0.9100990295410156, 1.3514270782470703, -0.08342576771974564, 1.285004734992981, 1.5752990245819092, -1.4010474681854248, -1.2761058807373047, -1.3662362098693848, -0.9143948554992676, 0.8858084678649902, -0.7797817587852478, -0.6043192148208618, -0.08563406020402908, 0.35116276144981384, -0.8874192237854004, 0.5603712797164917, -1.968072772026062, 2.111436128616333, 1.549048662185669, 0.5137109160423279, 1.3288768529891968, 0.5071602463722229, -0.07063170522451401, -0.7022562623023987, 1.0996296405792236, -1.9097084999084473, 0.8022359013557434, -0.013925585895776749, 2.7757136821746826, -0.6290720701217651, 1.3670330047607422, 1.3258243799209595, -1.2349088191986084, 0.3477539122104645, -1.3141182661056519, -0.7050698399543762, 0.4377737045288086, -1.1194572448730469, 0.04019131138920784, -0.4999024569988251, -1.240010142326355, 1.0242218971252441, -0.6513067483901978, -0.16594010591506958, 0.686766505241394, -2.2024998664855957, -2.4910428524017334, 0.6872307658195496, 0.770619809627533, 0.7064372897148132, -0.44290316104888916, 0.7066067457199097, -0.9226403832435608, 1.546994686126709, -0.4892587661743164, 0.884900689125061, -1.3764864206314087, 1.1387087106704712, 0.5874446034431458, -1.2834458351135254, 0.6416445374488831, -3.101109743118286, -0.8007678985595703, 0.7524053454399109, 0.10185443609952927, 0.027618613094091415, -1.3942172527313232, -0.994401216506958, -0.44296133518218994, 1.4742413759231567, 1.1063828468322754, 0.09890404343605042, -0.11584648489952087, 0.7975367307662964, 0.07929161190986633, 0.5764704942703247, 1.3848603963851929, 0.15067775547504425, -0.18193423748016357, 0.0733843445777893, 0.6340798735618591, 0.9568007588386536, 0.26474514603614807, 0.20277370512485504, -0.24461670219898224, -0.1181565374135971, -0.65812087059021, 1.5372726917266846, 1.3258861303329468, -0.5320649147033691, -0.859528660774231, 1.305681824684143, 1.8996217250823975, -0.31638336181640625, 0.13084560632705688, 0.9268977642059326, 1.0442588329315186, 2.5587100982666016, 0.3385021388530731, -0.48840394616127014, 1.7785735130310059, 0.5835737586021423, -0.25307247042655945, -0.39690908789634705, -0.1990446001291275, 0.42807257175445557, 0.5201990008354187, -0.3603871464729309, -0.06698223948478699, -1.0061132907867432, 1.181592345237732, 1.3450217247009277, -0.21436436474323273, -1.23899507522583, 0.4400429427623749, -0.9661295413970947, 0.5911432504653931, 0.11665727943181992, -0.5077493786811829, -0.40072929859161377, 0.3966461420059204, -0.48784711956977844, -1.3227158784866333, 0.4483719766139984, 1.404581069946289, 0.337484747171402, 0.09603416919708252, 0.6724634766578674, 1.352687120437622, -1.0251985788345337, -0.603012204170227, -0.4632694721221924, 0.28420209884643555, -0.38061538338661194, -0.12084973603487015, -0.4574846029281616, -0.8303455114364624, -0.00643036188557744, -0.06514519453048706, 1.2788722515106201, 0.6861264705657959, 0.36281850934028625, -1.486192226409912, 0.24634762108325958, -1.4410091638565063, -1.448938250541687, -2.266460418701172, -0.500734269618988, 0.1880030333995819, 0.46367162466049194, 0.5422528982162476, 1.7144839763641357, 0.5732307434082031, 0.44019296765327454, 0.07365518063306808, -0.6830049157142639, 1.4042186737060547, 0.4370102882385254, -0.04853997007012367, -0.2617647349834442, -2.09586238861084, 0.5132119655609131, 0.13563980162143707, -1.5813199281692505, 1.6527820825576782, 0.2722797691822052, -1.0758659839630127, -0.10390304774045944, -0.40296319127082825, -1.4731935262680054, 0.9434873461723328, -0.6853531002998352, 0.6820791959762573, 0.02611268311738968, -1.6755127906799316, -0.7171502113342285, -0.5161948800086975, 1.2049607038497925, 1.7098976373672485, -0.020264500752091408, 0.8567662835121155, 1.3697727918624878, 1.2192323207855225, 0.07423160970211029, -0.6906955242156982, -0.19460196793079376, 0.11593510955572128, -0.32519182562828064, 1.7345561981201172, -1.113079309463501, 0.5309948921203613, -0.0405491441488266, -1.372402548789978, 1.0845304727554321, -0.7135729193687439, 0.8184810280799866, -0.49033358693122864, 1.9315141439437866, 1.595679521560669, -1.0470223426818848, -0.8500853776931763, 0.9098020195960999, 0.5901104211807251, 0.03879223391413689, 0.4203962981700897, 0.3716655969619751, 0.06609673798084259, -0.7863838076591492, 1.0319961309432983, 0.06832130998373032, -0.8424659967422485, 0.0779288113117218, 1.3192192316055298, 0.027478912845253944, -1.540074348449707, -0.06723129004240036, -0.7271018028259277, 0.3326098620891571, -0.2690408229827881, 0.16111330687999725, 0.200912743806839, -0.37902840971946716, -1.112961769104004, 1.9915506839752197, 0.0841912105679512, 1.505113959312439, -2.3233120441436768, 1.1158771514892578, -1.6531497240066528, 1.5433465242385864, 0.6158214807510376, 1.0566997528076172, 1.0177028179168701, 0.41980475187301636, -0.2873194217681885, -0.520729660987854, -1.2221559286117554, -0.3840430676937103, 0.677828311920166, 0.010571726597845554, 0.3135465383529663, 0.1308371126651764, -0.20554842054843903, 1.0326125621795654, -1.0907599925994873, -1.3201823234558105, 1.6023669242858887, 0.8380061388015747, -0.6393977403640747, 1.997316837310791, -0.4205784201622009, 1.2063621282577515, 0.014989225193858147, 0.316805362701416, 0.8218562602996826, 0.7237688302993774, 1.0621144771575928, -0.5762221813201904, -0.800150990486145, -0.010281568393111229, 0.35744211077690125, 1.4537688493728638, 0.27508339285850525, -0.4575154185295105, 1.5681147575378418, -0.14151246845722198, -0.0478869304060936, -0.30805107951164246, -1.940348505973816, -1.4056718349456787, 1.049782156944275, 0.4825749397277832, 0.9318603873252869, 0.9666857719421387, -0.7068518400192261, -1.1998158693313599, -0.13665565848350525, -0.2185966521501541, 0.2627830505371094, 0.5815832018852234, 0.32900765538215637, -0.005527224857360125, 0.30632859468460083, -0.01649502106010914, -0.49482548236846924, 0.7485300898551941, -0.2807111442089081, 0.3642311990261078, -1.0541231632232666, 0.22050651907920837, -1.1685086488723755, 2.2025625705718994, -0.3586149215698242, -0.7444276213645935, -1.7086420059204102, -0.9765869379043579, 1.6208258867263794, 1.920255184173584, 0.12283705174922943, 0.3167804181575775, -0.3988550305366516, 0.6477174758911133, -1.0487502813339233, 0.006223244592547417, -0.1862906664609909, -0.9295494556427002, -0.5277168154716492, 1.1433382034301758, 0.27497708797454834, 1.1690232753753662, 0.3160228133201599, -0.07612936198711395, 0.9825711250305176, -2.044318914413452, 0.4326530396938324, -0.10692974925041199, -0.7093307375907898, 1.2665982246398926, 0.7081845998764038, -0.33439013361930847, -0.24711717665195465, -0.24926036596298218, 0.3857600688934326, -0.3774004578590393, 0.08085876703262329, 1.209826946258545, 0.434769868850708, 0.854123055934906, -0.5347198843955994, -0.3533077836036682, -0.5656448602676392, 0.024009188637137413, 0.20447993278503418, -1.474791169166565, 2.90061092376709, 2.329117774963379, -1.5249838829040527, 1.1713908910751343, -0.25583264231681824, -0.5741105079650879, -0.5653674602508545, -1.172859787940979, -0.3639712333679199, -0.5810875296592712, -1.1281615495681763, -0.4608311057090759, 0.024402108043432236, -1.37697434425354, 0.3907180428504944, -0.2124403864145279, -0.27205154299736023, -0.4100101590156555, -1.5525290966033936, 0.7886167168617249, 0.6872934103012085, -0.6391369700431824, -0.9805509448051453, -0.8342499732971191, 0.06039979308843613, 0.4913744628429413, 0.08809557557106018, 0.47765830159187317, -0.08939386159181595, 0.3405997157096863, 1.7360141277313232, 0.24624145030975342, -1.2369719743728638, -0.06139218807220459, 1.2291206121444702, -1.2038860321044922, 0.35306450724601746, 1.3803362846374512, 0.5387757420539856, 1.170657753944397, 1.9282008409500122, 0.5731423497200012, -0.5430763363838196, -1.1404831409454346, 1.7334880828857422, -0.006214366294443607, 0.4509076476097107, 1.4652042388916016, -0.7103082537651062, 0.180085688829422, -0.5302600264549255, 0.7535815238952637, 1.648524284362793, -0.8973758816719055, -0.970747709274292, 1.7526962757110596, 0.3786667585372925, 0.7081519365310669, 1.7624238729476929, 0.5079694986343384, -0.09019612520933151, 0.6882871389389038, -0.3473051190376282, 0.6322211623191833, -0.24403993785381317, 0.5372968912124634, -0.7780193090438843, -0.6023900508880615, -0.18810191750526428, 0.5973517894744873, 1.718924641609192, 0.48867136240005493, -0.5114132165908813, -1.2288029193878174, -0.43033942580223083, -1.2166774272918701, 0.9896777272224426, -1.0326929092407227, -0.19998043775558472, 0.8997708559036255, -0.36717891693115234, 0.5158312916755676, 2.3074686527252197, -0.6399033069610596, 0.626378059387207, 0.6489725112915039, -0.055201172828674316, -0.6468269228935242, 0.7971866130828857, -0.33063745498657227, 0.17120638489723206, 2.0862584114074707, -0.9616897106170654, -1.044224739074707, 1.7012856006622314, -0.6053822040557861, -2.0192947387695312, 1.100230097770691, -0.5318860411643982, -0.33340924978256226, 0.3439268171787262, 0.0742354616522789, -0.9161642789840698, -0.3534911870956421, 1.191125750541687, 0.6077690124511719, -1.8987382650375366, 0.03340473398566246, -1.5817705392837524, -1.120652675628662, -0.5010656714439392, 1.5818662643432617, 0.6541739702224731, -1.0823230743408203, 1.1517173051834106, 0.056449372321367264, 1.3468539714813232, 0.10310742259025574, -1.038936734199524, 0.3974250257015228, 1.8840694427490234, -0.09520252794027328, 0.5978793501853943, -0.0805952399969101, -1.116867184638977, 0.4546949565410614, 0.34207767248153687, -0.6044734716415405, 0.1968822330236435, -0.5211434364318848, 1.0718755722045898, 0.253601610660553, -1.1778520345687866, 1.499542474746704, -1.652574896812439, -0.8526055812835693, 0.2877039909362793, -0.22397972643375397, 0.7354719638824463, -0.4381621181964874, 1.0249912738800049, 1.2321550846099854, 0.565506637096405, 0.7115970253944397, -0.9815586805343628, 2.4715826511383057, -0.046194177120923996, -0.4494222402572632, -1.2067188024520874, 0.1935436874628067, 0.04624268040060997, -1.0985634326934814, -0.8840369582176208, 0.45582762360572815, -1.0530731678009033, -1.8419092893600464, -0.32351258397102356, 0.5803203582763672, 1.1441277265548706, 0.44750380516052246, -1.473020315170288, -0.3369464576244354, -0.6864138841629028, 0.4385101795196533, -1.4591245651245117, -0.08957706391811371, 1.2276605367660522, 0.5276268720626831, 0.0743338018655777, 0.7195516228675842, 0.2472168505191803, 0.8313308954238892, 0.5728986263275146, 0.21861955523490906, 0.08123613148927689, -0.580187976360321, 0.8204171061515808, 0.36142492294311523, -0.3140605092048645, 1.0371253490447998, 0.7168887257575989, -1.257134199142456, -0.4336567521095276, -1.0475399494171143, -0.06676094979047775, -1.369200587272644, -0.5695279836654663, 0.5321061611175537, -0.11964252591133118, 0.06719199568033218, 1.3270517587661743, 1.6649514436721802, 0.7808016538619995, 0.6588834524154663, -0.2428256869316101, 0.057753488421440125, 1.3103649616241455, -2.0837244987487793, -0.6432774662971497, -0.22661584615707397, 1.0165719985961914, -0.7235369086265564, 1.339245319366455, 1.6526466608047485, 0.14902035892009735, -0.9516107439994812, -1.29148268699646, 0.30331605672836304, 1.7797789573669434, -0.8386068940162659, 1.4634554386138916, 1.1607977151870728, -1.3707842826843262, -0.9157952070236206, 0.7310544848442078, 0.9757773280143738, 1.8544721603393555, -0.5591349601745605, 0.022705014795064926, 0.6285997033119202, 1.937678337097168, 1.0176492929458618, 1.1334670782089233, -0.8324664831161499, 0.5425337553024292, -0.33413857221603394, -1.1524596214294434, 0.3382393419742584, 3.5010039806365967, 1.6508331298828125, -0.13156139850616455, 0.9457908272743225, 2.1580851078033447, 0.8223727345466614, 2.456084966659546, 1.5839284658432007, -2.5170364379882812, 0.7147006392478943, 0.20967552065849304, -0.7703407406806946, -0.6241031885147095, 0.3089378774166107, 0.349133163690567, 0.3415059447288513, 1.7828150987625122, 0.30461519956588745, 0.312697172164917, 0.3198961913585663, -0.2774587869644165, 0.1993558704853058, 3.1867332458496094, -2.3671793937683105, -0.010283228009939194, 0.7732049226760864, -0.1982564479112625, 1.1393847465515137, 0.21755795180797577, 0.8549681901931763, 1.2225786447525024, -0.6645573377609253, 0.4254510998725891, -0.08950310945510864, 0.5050090551376343, -0.03623463585972786, 0.5519239902496338, -0.5033851265907288, -0.27855655550956726, 0.38980069756507874, 0.035774074494838715, 0.1498911827802658, -1.0282671451568604, -0.867449939250946, 0.8025915622711182, -0.7021177411079407, 0.2798005938529968, -1.104779601097107, -0.24599486589431763, 0.3479064404964447, -1.2004964351654053, -1.3996081352233887, -0.5387610197067261, -0.8746501803398132, -1.7647464275360107, -0.9309567213058472, -0.1938074380159378, 0.11704042553901672, -0.15973100066184998, 0.30411282181739807, -0.3320196270942688, 0.9326328039169312, 0.060366056859493256, 0.9672592282295227, 0.5480862855911255, 0.09853044897317886, 1.039167881011963, -1.1768046617507935, -1.6363502740859985, 0.027737081050872803, -1.563022255897522, -0.24700263142585754, -2.1387407779693604, -0.27247071266174316, -1.443731665611267, 1.9272449016571045, -0.09468051046133041, -0.2458174228668213, 1.3270140886306763, 0.06033291667699814, 0.9184982776641846, -1.5788588523864746, 0.26675546169281006, 0.22636349499225616, -1.6825222969055176, -2.3698298931121826, 0.6132472157478333, -1.1601042747497559, -0.2310725301504135, 1.339656949043274, -0.998971164226532, -0.2945268452167511, -0.3804229497909546, -1.1507757902145386, 0.5998005270957947, -1.4413478374481201, 0.31014785170555115, 0.41358017921447754, -0.8308799862861633, 1.108995795249939, 0.8657708764076233, 0.23994942009449005, -0.6178882718086243, 0.6836771368980408, 0.003668291727080941, -0.996645987033844, 0.23994262516498566, -0.7230538725852966, -0.11815302073955536, -0.1194930151104927, 0.43493491411209106, 0.3680253028869629, -0.04804319888353348, 0.43584492802619934, 1.4624731540679932, -1.6122829914093018, -1.6360872983932495, -1.6082102060317993, -0.46938517689704895, 0.35593655705451965, -0.7388746738433838, 0.41371166706085205, 0.8929959535598755, -0.7957450747489929, -0.5573137998580933, 0.6066458225250244, 0.36220356822013855, 0.22464294731616974, 1.5761674642562866, -1.391496181488037, 0.8805974125862122, -0.7503166794776917, -0.8707498908042908, -0.45076149702072144, -0.20670782029628754, 0.5914629697799683, 0.5306383967399597, -0.5352937579154968, 1.5102263689041138, -1.4041579961776733, 1.696155071258545, 0.6598222255706787, 0.4918617308139801, 1.1036626100540161, 1.2307504415512085, 0.8120114207267761, -1.3616098165512085, -0.8850715756416321, 0.24706564843654633, 0.09206734597682953, 0.10013794898986816, -1.3402481079101562, -0.126215398311615, 1.5275002717971802, 0.20991560816764832, 1.250699758529663, -0.8202309608459473, 1.2924833297729492, 1.6392146348953247, -0.1846882849931717, 1.1624747514724731, 0.07510357350111008, 0.5880265831947327, 0.26955845952033997, -0.26728612184524536, -0.3529808521270752, 0.06713647395372391, -0.48040664196014404, -1.010591983795166, -1.2337815761566162, 0.19636939465999603, 0.7371436953544617, 0.6203107833862305, 1.5077426433563232, -1.4461328983306885, 0.010622814297676086, 0.6955443620681763, -0.7019912600517273, 1.3909201622009277, -0.6046265363693237, 2.1528728008270264, -0.6146133542060852, -1.2648066282272339, -0.09403906762599945, 1.1422383785247803, -0.9175623655319214, 0.34690696001052856, -0.9682802557945251, -1.6887022256851196, -0.24015894532203674, 0.9131070971488953, 0.10373333096504211, -0.2722945511341095, -1.8293460607528687, 0.031184161081910133, 0.642490804195404, 1.9860010147094727, 0.1207309365272522, 0.42005881667137146, -0.7242626547813416, -0.8673815131187439, 1.2841061353683472, 0.8138177394866943, -0.04393845796585083, -0.12414475530385971, 0.010800987482070923, -0.8301957249641418, -0.3283281922340393, 1.0002672672271729, -1.9702117443084717, 1.159455418586731, 1.2082542181015015, 0.2820504307746887, 1.519537329673767, -1.6199852228164673, -1.0711649656295776, -0.9532658457756042, -0.5841324925422668, -0.8960805535316467, -1.5629647970199585, -0.698070228099823, -0.9764760732650757, -1.4588114023208618, 0.4265071153640747, 1.4407939910888672, 0.24293695390224457, 0.1508392095565796, 0.25234493613243103, 1.6428172588348389, 0.49113529920578003, 0.4734790325164795, -0.006588586140424013, -1.0960527658462524, 1.267188310623169, 1.2707629203796387, 0.2955610454082489, 0.13242818415164948, 1.3052133321762085, 0.9520296454429626, -0.3267824947834015, -0.2943064868450165, 0.878911554813385, 1.0112122297286987, -0.7454066276550293, 2.2106335163116455, -1.177549123764038, -0.765296995639801, -0.24265477061271667, 0.7456341981887817, -0.1724376529455185, -0.6823351383209229, 0.8989181518554688, -0.10842015594244003, 0.26273298263549805, 0.43871915340423584, -0.9013411402702332, -0.9042394161224365, -0.021202554926276207, -0.394488126039505, -0.35993582010269165, 0.35389816761016846, -0.38593289256095886, -1.0116554498672485, 1.2758044004440308, -0.5393763184547424, 0.13974027335643768, -1.2895654439926147, 0.5644819736480713, -0.9830734133720398, 0.06247546896338463, 1.0420734882354736, -0.09304441511631012, -0.2717325985431671, 1.5145474672317505, 0.6789610385894775, 1.1259599924087524, -1.061535120010376, -0.11298073083162308, -0.07079287618398666, 1.8551877737045288, 0.9832988381385803, -0.6462927460670471, 0.5935477018356323, -0.09548377245664597, 0.20879802107810974, -0.7504525780677795, 0.7685794830322266, 0.5513703227043152, -0.17322561144828796, -0.04736337438225746, 1.7705551385879517, -0.15710505843162537, 1.6413434743881226, -0.7487513422966003, -1.5012520551681519, -0.6732559204101562, -1.0588197708129883, 0.40829765796661377, 0.08994857966899872, -1.0984413623809814, -0.36129873991012573, -1.0080080032348633, -0.14311298727989197, 1.999056100845337, 1.252693772315979, -1.4315834045410156, 0.4421485364437103, -0.1588086187839508, -1.4509364366531372, 0.7004755735397339, -1.9905028343200684, -0.6897918581962585, -0.6587681174278259, 0.10123598575592041, -1.3456921577453613, 0.1628938913345337, -1.3364224433898926, 1.3218191862106323, 1.4507558345794678, 0.35675665736198425, -3.4236338138580322, 2.253627061843872, -1.8351062536239624, 0.05667807161808014, 1.9317339658737183, -0.28135696053504944, -0.7408050894737244, 0.3776063024997711, -1.350504994392395, -0.5069355964660645, -0.3451196253299713, 0.5779469013214111, -0.6420303583145142, -0.3066166341304779, -1.4222134351730347, -0.03860660642385483, 0.8931888341903687, -1.0270121097564697, 1.0226191282272339, 1.739721417427063, 0.8013507723808289, 0.5658381581306458, 1.922380805015564, 1.0734058618545532, -0.09988537430763245, 0.5678088068962097, 0.4746842682361603, 0.9334282279014587, -1.1962134838104248, -1.5701626539230347, -0.7965176701545715, -0.8817328810691833, 0.8414355516433716, 2.0199103355407715, -0.6556959748268127, -0.06595878303050995, -0.07424778491258621, 0.5164698958396912, 0.8098201751708984, 0.5815158486366272, -1.1885817050933838, -0.505105197429657, -0.5568352341651917, 0.36347195506095886, 0.5344389081001282, 0.8904293179512024, -0.3862038850784302, -0.2629832923412323, -1.0163099765777588, 1.1163393259048462, 0.6376516222953796, 0.19756169617176056, 1.5009043216705322, 0.024014944210648537, 0.15921351313591003, 0.2627536654472351, 0.7061830759048462, 1.2647813558578491, 0.1498023122549057, -0.18484506011009216, 1.0084493160247803, 0.410754919052124, -1.2348498106002808, -1.068015694618225, 0.8725427985191345, 1.4568918943405151, 0.6712451577186584, -0.024982474744319916, -0.050125230103731155, -0.39634737372398376, -1.0487850904464722, -0.7399082183837891, 0.028794260695576668, -0.8659921288490295, 1.2263449430465698, 0.1748659461736679, -0.007868042215704918, -1.6359102725982666, 0.051123276352882385, -0.4592973291873932, 0.8198802471160889, -1.4218535423278809, -1.8510280847549438, -0.18289203941822052, -0.2082919031381607, -0.28113076090812683, 0.9345863461494446, -0.642219066619873, -0.14484785497188568, 0.8670397996902466, 2.390751361846924, 1.4494373798370361, -0.19272543489933014, 0.19444650411605835, 1.2834007740020752, 1.5905219316482544, 0.565505862236023, 0.46899738907814026, -1.3724387884140015, -0.6886653900146484, 1.0060619115829468, 0.2371249943971634, -0.22856631875038147, -0.512066662311554, 0.37310856580734253, -0.505256175994873, 0.41470208764076233, -2.1554112434387207, 1.380152702331543, -0.258645236492157, 0.34363728761672974, 0.5571105480194092, 2.1528139114379883, -0.6460267305374146, -0.30565017461776733, -0.5763909220695496, 0.6134364604949951, -0.9208450317382812, 0.5171413421630859, 1.1879734992980957, 1.2711994647979736, -0.13910242915153503, 1.2582638263702393, -0.44810909032821655, -0.6147212386131287, 2.0536487102508545, 2.2312862873077393, -0.8296805620193481, -1.7009949684143066, -0.6313053369522095, 0.3632490634918213, 0.21774303913116455, 1.0981484651565552, -0.5650109052658081, 1.5801845788955688, 2.1150894165039062, -0.6750603318214417, 2.2633016109466553, -0.1413630098104477, -0.4896438419818878, 0.5627447366714478, 0.31032446026802063, 0.018024925142526627, 1.1818122863769531, -0.7114922404289246, 0.526465892791748, -0.8262901902198792, 0.5184200406074524, 0.03219066560268402, -0.06288893520832062, 0.5603020787239075, 0.10815463960170746, 0.6850743293762207, -2.0530343055725098, -1.3474183082580566, 0.46255242824554443, 0.4442182779312134, 1.325705647468567, 1.3725539445877075, -1.051188588142395, -0.4248499274253845, -1.8845469951629639, 0.379904180765152, -0.9506605267524719, 0.6928113102912903, 0.6047332286834717, -1.1951665878295898, 1.281992793083191, -0.34081944823265076, 0.5521399974822998, 0.691960871219635, 0.9596428871154785, -0.7568160891532898, 0.09570810943841934, -1.0011078119277954, 0.14537562429904938, 1.128495216369629, 0.7981462478637695, 0.36914893984794617, 0.48763445019721985, -1.190021276473999, -2.393338918685913, 0.9957087635993958, -2.136408805847168, 0.7625260949134827, -0.1530458778142929, -0.5133539438247681, -0.5997365713119507, -0.2310160994529724, 0.8377372026443481, -0.9695630073547363, -0.9889869093894958, -0.401690810918808, 1.46099853515625, 0.8843457102775574, -0.21773052215576172, -1.7873907089233398, 0.7458462715148926, 1.9239866733551025, 0.45698004961013794, -2.2904672622680664, 0.18625539541244507, 1.1431509256362915, -0.5015356540679932, -0.0037896991707384586, 0.018400058150291443, -0.2581934332847595, 0.4284288287162781, 1.50681734085083, 0.8430250287055969, 1.4650520086288452, 0.12808674573898315, 0.303672194480896, -1.4375064373016357, 1.0212844610214233, -0.11133186519145966, 0.2635957896709442, 0.8254147171974182, 0.021520325914025307, 1.8409467935562134, 0.8675661683082581, 0.1898559331893921, -0.8453050851821899, -1.245116949081421, 1.398729920387268, 1.7179476022720337, 0.1652430295944214, 1.2515337467193604, -2.7062137126922607, -0.5102187991142273, 0.610564649105072, 1.1983078718185425, 0.15021701157093048, -0.07065815478563309, -0.001475203549489379, 0.40881431102752686, -0.6929314732551575, -0.3579290807247162, -0.5133786201477051, 0.6404692530632019, 0.02323799580335617, -1.0739105939865112, -1.5247503519058228, 0.35228073596954346, -1.334106206893921, 0.4661732316017151, 2.246605634689331, -1.7288832664489746, 0.2414310872554779, 0.4075682461261749, -0.05644812062382698, 0.3706827759742737, 1.252099871635437, -0.37885767221450806, 0.05672838166356087, 1.1465853452682495, 0.1336359679698944, -1.4050534963607788, 0.3459082543849945, 0.6689968109130859, -0.750769853591919, -0.5958080291748047, -0.2643069922924042, -0.4683106243610382, -0.5030724406242371, -0.33684515953063965, 0.13875745236873627, 0.33470696210861206, 0.3116804361343384, 0.47149941325187683, 1.3890892267227173, 0.20460426807403564, -0.9856916069984436, -0.0931883230805397, -1.7605206966400146, 0.8672109246253967, 1.9250686168670654, -0.7595016360282898, 1.2441712617874146, 0.7487488985061646, -0.2004127949476242, -0.09874843060970306, -0.49112778902053833, 0.1227906122803688, -0.4214593768119812, 1.838207721710205, -0.828363835811615, 1.0487635135650635, 1.2886704206466675, -1.3034188747406006, -0.3774552643299103, 1.7753102779388428, 0.4977693259716034, 0.8582208156585693, 0.7865264415740967, -0.4606183171272278, 0.01186515111476183, -2.258902072906494, 0.9024047255516052, 0.7642454504966736, 0.7796726822853088, 0.6488118767738342, 0.8133699893951416, -1.221071481704712, -1.5743699073791504, -0.5234638452529907, -0.9844873547554016, 1.1396567821502686, -0.5373557806015015, 0.3674285411834717, -1.0758365392684937, 1.247477650642395, 1.1953511238098145, -2.203723907470703, 0.460462749004364, -0.314120352268219, 0.03803668171167374, -0.042075153440237045, -0.6183479428291321, 1.2509348392486572, -0.33068087697029114, -1.4919942617416382, -0.03282584249973297, 0.16557800769805908, -1.7753374576568604, -0.19350892305374146, -0.06435851007699966, 0.1917019933462143, 0.49516746401786804, -0.554619550704956, 0.7369165420532227, 0.2886049151420593, -1.0687880516052246, -0.24772052466869354, 0.43951672315597534, -0.34090456366539, 0.26410993933677673, -1.3168153762817383, -1.0904713869094849, 1.1321014165878296, 0.34739843010902405, -0.22675451636314392, 0.17662005126476288, 0.7874548435211182, 0.5927537679672241, 0.2702954411506653, -0.5092017650604248, -1.8508139848709106, -0.09550915658473969, -0.21872137486934662, 1.5354357957839966, 0.38933101296424866, 1.2400474548339844, -0.5194528698921204, 0.4548552334308624, 0.31518107652664185, -0.7258563041687012, -0.4925777316093445, -0.6722643375396729, 1.2026690244674683, 0.6130151152610779, -1.1712517738342285, -1.3403090238571167, 0.44614478945732117, -0.4171799421310425, 0.05902581289410591, 1.2624804973602295, 1.059501051902771, 0.8220215439796448, -0.4516347348690033, 0.17300638556480408, 0.8300368785858154, 0.04143516346812248, -0.7470070123672485, -1.0754770040512085, 0.3786233961582184, -0.7808800339698792, -0.22891739010810852, -0.6521589159965515, 0.8678596615791321, 1.1229993104934692, 2.001966953277588, -1.225235939025879, 0.5635823011398315, 0.7044203877449036, -1.3782284259796143, 1.32588529586792, 2.0257067680358887, 1.2741137742996216, -0.25256383419036865, -1.283663272857666, 0.10093493014574051, 1.29982590675354, 0.5625365972518921, -0.014101183973252773, 1.6056803464889526, 0.8367495536804199, 1.1363986730575562, -0.7569200992584229, 1.0305997133255005, -0.24885456264019012, 0.3966379165649414, -0.1989043951034546, 0.3438446819782257, -0.21195146441459656, 0.3017890155315399, -2.321072578430176, -1.264488697052002, 1.9290868043899536, -0.5477479696273804, -0.49525225162506104, -0.06929110735654831, -2.0452470779418945, 0.40792685747146606, -0.6307077407836914, 0.3782849907875061, 2.328929901123047, -0.3638041615486145, -0.8635632395744324, 1.4250524044036865, -0.45750683546066284, 1.1730906963348389, 0.7197269797325134, 0.7297815680503845, 0.10251665860414505, -1.0449317693710327, -1.589185118675232, -0.16058821976184845, -1.3973807096481323, -0.4115188717842102, -1.0565922260284424, -0.4676722586154938, -0.19181780517101288, 0.48765701055526733, -0.40945541858673096, 0.2991310954093933, 0.883743166923523, -0.3627433776855469, 1.0232468843460083, 0.8134499192237854, -1.6097168922424316, -0.09695509076118469, 1.613703727722168, 1.0605463981628418, -0.7468411922454834, 2.0076184272766113, -0.5846594572067261, -0.9591276049613953, 1.194774866104126, -0.9053308963775635, -0.3777103126049042, -0.27279138565063477, 0.9488219022750854, 0.23754307627677917, 0.6977294683456421, -0.029920252040028572, -1.7185477018356323, 0.4046296179294586, -0.4921787679195404, 0.15953020751476288, -0.34108105301856995, 0.8841574788093567, -0.1496691256761551, -1.636427402496338, -0.8428424000740051, -0.6781303286552429, -0.8603988885879517, 2.45405912399292, -0.9583150744438171, -0.5527498126029968, -0.8199004530906677, -1.5800203084945679, 0.3638284504413605, -0.4147149324417114, 1.2819350957870483, 1.4604345560073853, -0.6200644969940186, -2.1394290924072266, -0.5457748770713806, 0.7119128704071045, -0.4035010039806366, -0.4507066607475281, 0.17627523839473724, -0.05141029506921768, -0.7875949740409851, 0.628493070602417, -0.9743261337280273, 1.2369284629821777, 1.813940167427063, 1.3219153881072998, -0.9080224633216858, -1.0359238386154175, 0.698412299156189, 0.17583401501178741, -1.0256942510604858, -0.06930892169475555, -0.37662896513938904, -0.0881766676902771, 0.3280309736728668, -0.46739187836647034, -0.8292897939682007, -2.1337485313415527, 1.3523600101470947, -0.9910383224487305, 1.7519075870513916, -0.03738725930452347, 0.3599800765514374, -0.3488122820854187, 0.04283464699983597, 0.7458696365356445, -2.211702585220337, 1.5717567205429077, 0.9193817973136902, -1.4205379486083984, 1.0909075736999512, -0.8793131709098816, -0.15181882679462433, 0.08085867762565613, -0.9316440224647522, -0.5321239233016968, -0.6042446494102478, -0.7651544213294983, -1.7920323610305786, 1.5836104154586792, 0.8260266184806824, -0.6100199818611145, -0.7680028080940247, 1.2971255779266357, 0.8383170366287231, -0.42664140462875366, -0.2914358973503113, 0.47223740816116333, 0.5059854388237, 0.08278635144233704, 0.32814884185791016, -0.27074405550956726, -1.1584889888763428, 0.421294629573822, 0.7079086899757385, 0.1066007986664772, -0.4921809434890747, -0.5754324197769165, -0.5231444239616394, 0.7742962837219238, -0.18922768533229828, 1.3717314004898071, -0.1413632482290268, -1.0649967193603516, 0.5773984789848328, -0.05772055312991142, -1.2873222827911377, 0.5105907917022705, 0.568557620048523, 1.2887667417526245, 0.49321219325065613, -0.5401250720024109, 0.44444751739501953, -0.6095677018165588, 0.9644752740859985, 1.484740138053894, 0.010921473614871502, -1.8125951290130615, -0.022373592481017113, 0.2742646336555481, 0.27574101090431213, -0.830985963344574, 1.1869715452194214, 1.190744400024414, -2.0655243396759033, 0.09256156533956528, 0.8264639377593994, 1.0064541101455688, 0.3899169862270355, 0.9950791597366333, -0.7495323419570923, -0.7814085483551025, -0.49897122383117676, 0.5975967645645142, 1.2217791080474854, -0.04016202315688133, -0.11907543987035751, 0.19234056770801544, -0.6770980358123779, 0.3193354308605194, 0.2708011269569397, 0.71757572889328, -0.44310060143470764, 0.8427134156227112, -0.09504082053899765, -0.3845362365245819, -0.88368821144104, 0.7475668787956238, -0.08368346095085144, -0.008312596008181572, 0.07918122410774231, -1.5581247806549072, 0.01939338445663452, -0.08850588649511337, 1.1124986410140991, 0.7253099083900452, 0.5909878015518188, 2.169069766998291, -0.29758960008621216, -0.6431765556335449, -0.03667505457997322, -0.5312623977661133, -0.7182332873344421, 2.222632884979248, 1.7747853994369507, -0.041322674602270126, -0.24367354810237885, -1.269744634628296, 0.17091743648052216, -0.5293787717819214, -1.1045262813568115, -0.09418700635433197, -0.12026920169591904, -0.4941534399986267, -1.4219352006912231, 0.36256036162376404, -0.8052845001220703, -0.9678937196731567, 0.688748300075531, -1.4770179986953735, -0.7931230068206787, 0.45388442277908325, 0.5196749567985535, 0.07932479679584503, -1.6099002361297607, -0.6924731135368347, 0.29633277654647827, -0.9770949482917786, -1.282077431678772, -0.7161006927490234, -1.0855472087860107, 0.0042358883656561375, -0.8397420048713684, -0.27342087030410767, -0.6112993359565735, 1.0340787172317505, 0.8216263055801392, -0.3853000998497009, -1.5674717426300049, 0.3913840651512146, 1.7960761785507202, -0.36464306712150574, -0.4086468517780304, -1.0519884824752808, 1.004040002822876, 0.7918250560760498, 0.24033600091934204, 1.7559199333190918, -1.4793825149536133, -2.4356796741485596, 0.9504199028015137, 0.3274521827697754, 0.35552316904067993, 0.3293660879135132, -0.39724084734916687, -1.7255966663360596, -0.6219899654388428, 0.42086637020111084, 1.1799408197402954, -0.21210676431655884, 1.3441227674484253, 0.18653960525989532, -1.093782663345337, -0.8838498592376709, -2.5041401386260986, 0.3743944466114044, -0.6786481142044067, -0.31497082114219666, 1.432262897491455, -2.2167809009552, -1.12917160987854, -0.5610722899436951, -0.7265071868896484, 1.1979387998580933, 0.9375250339508057, 0.3220018446445465, -0.30695778131484985, 0.036870554089546204, -0.7259985208511353, 0.9893746972084045, 1.1796720027923584, 0.9237304925918579, -1.1429487466812134, 0.6974831223487854, -0.18486745655536652, 1.6567516326904297, 0.9128431677818298, -0.02203953266143799, -0.7260380387306213, 0.20342285931110382, -0.7054110765457153, -0.6639354825019836, 0.4910569190979004, -0.13552138209342957, -0.24333971738815308, -1.7665901184082031, -1.7208902835845947, -1.5254241228103638, -1.6276296377182007, 0.42401549220085144, 0.5585992932319641, -0.15287022292613983, -1.7047301530838013, -1.0235215425491333, -0.3511417806148529, -0.1752181351184845, -0.374721497297287, 2.382222890853882, 0.1315414160490036, 0.8232066631317139, 1.7764209508895874, -1.5055075883865356, -1.5770444869995117, 0.5856661796569824, -1.2017490863800049, 0.45967376232147217, 0.973457932472229, -0.508135974407196, 0.0672798827290535, -0.012203209102153778, 1.437164306640625, -0.07228866964578629, -0.24972862005233765, 0.2123885601758957, 0.04831039160490036, 0.022278405725955963, -0.06893280893564224, 1.3698513507843018, -0.5680445432662964, 0.15441274642944336, -0.24643076956272125, 0.333558052778244, -1.802191138267517, 0.8976401090621948, -0.7237213253974915, 0.04187455400824547, 2.235781669616699, -1.1946662664413452, 0.2292066514492035, -0.42047008872032166, -0.30428674817085266, -0.3399118185043335, -0.41244596242904663, 1.571059226989746, -0.8306007385253906, 1.8036242723464966, 0.3265196681022644, -0.8552032113075256, -1.466348648071289, 0.9366782307624817, -0.01823963224887848, 0.8112977147102356, 1.803064227104187, -0.7363269329071045, -0.320826917886734, -0.30629321932792664, -1.3547172546386719, -0.1464865356683731, -1.355048656463623, -0.8068174719810486, -0.720642626285553, -0.7579324841499329, -0.040192488580942154, 0.9099517464637756, 0.23769886791706085, -2.334106922149658, 0.9291107654571533, -0.5002602338790894, -0.7129952907562256, 0.6809146404266357, -1.16559898853302, -1.0336095094680786, -0.319094181060791, 1.2466320991516113, -0.06358101963996887, 1.0880286693572998, 0.12857778370380402, 0.18898174166679382, -0.901678204536438, -0.25229206681251526, -0.7578758597373962, -0.3770519495010376, 0.8687453269958496, 0.5045630931854248, 0.07701743394136429, 0.7555031776428223, -0.863705039024353, 2.4888603687286377, 1.402861475944519, 0.2207067310810089, -0.22285975515842438, 0.32157906889915466, -0.5219932794570923, -0.3682592511177063, -1.0534721612930298, -0.2660396695137024, -0.2205522209405899, 1.64679753780365, 0.23835507035255432, -1.1242001056671143, -0.4653911888599396, 0.4114038050174713, 0.5202413201332092, 1.3079757690429688, 1.450198769569397, 0.5494288206100464, 0.023461351171135902, -1.2185611724853516, 0.7593844532966614, 0.0802643895149231, -0.3482900559902191, -0.4557749927043915, 0.6905508041381836, 0.6298802495002747, -0.8017709851264954, 1.7749695777893066, 0.4708850383758545, -1.5843758583068848, 2.253615379333496, 0.014363103546202183, -0.4974907636642456, -0.40130311250686646, 0.8870514631271362, -0.5858309268951416, -0.2769627869129181, -1.9345200061798096, 0.02568868361413479, -0.32332485914230347, -0.2077236771583557, 1.3050668239593506, 0.12359460443258286, 0.05687255784869194, -0.1693696826696396, -0.3239741623401642, -0.2807133197784424, 1.3889497518539429, 1.3236441612243652, -0.028989048674702644, 0.5821642875671387, 1.6621601581573486, 0.39784854650497437, -0.03854995220899582, 0.17263232171535492, -0.8760885000228882, 0.8087530136108398, -0.7542358040809631, -1.2262557744979858, 0.9644199013710022, -1.3965015411376953, -0.5129486322402954, -1.5454498529434204, -1.6341196298599243, 0.8101987838745117, -1.2244727611541748, 0.6633430123329163, -0.7099289894104004, -0.3505921959877014, 0.20968490839004517, 1.2709473371505737, 0.27672427892684937, -0.8060691356658936, 0.5178765058517456, 0.7511399984359741, 1.358361005783081, -0.413939505815506, -0.5969024896621704, -1.0908989906311035, -0.8481667637825012, 1.1970402002334595, 1.2982916831970215, -1.1363919973373413, 0.7513997554779053, -2.154383420944214, -0.20082245767116547, -1.5257508754730225, -0.9127528667449951, 1.6275482177734375, 0.3045334219932556, 0.08305148035287857, 1.1663554906845093, -0.7032305002212524, 0.6994603872299194, 0.5306916832923889, 1.1080058813095093, 0.2302379459142685, -0.5365950465202332, 0.16897615790367126, -1.4565435647964478, 1.094191312789917, 1.126351237297058, 0.1325361281633377, -0.18113520741462708, 0.701309859752655, 0.861515998840332, 0.4222716689109802, -1.6056212186813354, 1.3935850858688354, 0.3251105844974518, -0.8632166981697083, -0.3561514914035797, -0.8856997489929199, 0.7178762555122375, -0.7523510456085205, 0.5190382599830627, 0.01993430219590664, 0.20639468729496002, 0.9517233967781067, 0.4356643855571747, 0.5520609617233276, -0.9163179397583008, -0.6444816589355469, -1.8293544054031372, 1.240525245666504, -0.9994257092475891, -0.5986261963844299, -1.0176200866699219, -0.6156457662582397, 1.3950610160827637, -1.390755534172058, -0.26501020789146423, -1.0518064498901367, -0.266286700963974, 0.13315008580684662, 0.34902751445770264, -0.23185928165912628, 0.26752766966819763, -0.7015582919120789, 2.247518539428711, 0.6579428315162659, 0.5904129147529602, -1.1060307025909424, -0.9746491312980652, 0.05489156022667885, 0.6480618715286255, 1.340970516204834, -0.16985593736171722, -1.725633144378662, 0.6055068969726562, -1.2222700119018555, 0.3298934996128082, 0.5702513456344604, 0.9813571572303772, 0.9518198370933533, -0.36096423864364624, 0.7641870975494385, -0.004331989213824272, 1.915831208229065, -0.43373405933380127, -0.09196159988641739, 1.7233459949493408, -0.9636831283569336, 0.8815656900405884, -1.5790088176727295, 0.8420231938362122, -1.265166163444519, 1.851220965385437, 0.893321692943573, 0.6345610022544861, 1.3964855670928955, -1.1119022369384766, 0.5063714981079102, 0.14402495324611664, 0.17820508778095245, -0.8973730206489563, 0.24238398671150208, 1.026639461517334, -0.40051281452178955, -1.3556404113769531, -2.019325017929077, 1.324440598487854, -1.2483071088790894, 0.16897007822990417, 2.1348204612731934, 0.1996062844991684, 1.2485604286193848, 0.3513805866241455, 0.5062078833580017, 0.7494341731071472, -0.13499581813812256, 0.6243245601654053, 0.7485663890838623, -0.9223400354385376, 0.5042790174484253, 0.5506026744842529, 0.6015239953994751, 0.02874166890978813, -0.2813514173030853, -0.7251843810081482, -0.5822994709014893, 0.3597530126571655, 0.7520111203193665, -0.8758552074432373, -0.9836949706077576, 0.8199852108955383, 1.5934265851974487, -0.3994028568267822, 0.47192203998565674, 1.5500884056091309, -1.2476634979248047, 0.8216185569763184, 0.7232509851455688, 1.1840776205062866, 0.374752938747406, 1.1589691638946533, -0.8610861301422119, -1.0777605772018433, 0.4579745829105377, -0.6627838015556335, 0.6097348928451538, 1.1840471029281616, -0.03211564943194389, 0.8777114152908325, 0.046235520392656326, -0.27666816115379333, -0.2832558751106262, 0.6043227910995483, 2.037501096725464, -0.6315439343452454, -0.9388247728347778, -0.17190372943878174, -1.6117182970046997, 0.7226200103759766, 1.7173206806182861, -1.2219855785369873, -0.005205733701586723, -1.2388203144073486, 0.20751594007015228, 0.05152913182973862, 1.4247609376907349, 0.17630192637443542, 1.1411609649658203, -0.788169264793396, 0.8966711163520813, -0.16588090360164642, -1.5016400814056396, -0.22264133393764496, -0.14256194233894348, -0.6397427320480347, 0.13357862830162048, -0.845375657081604, 0.36813127994537354, -1.364767074584961, 1.3529640436172485, 0.007451964542269707, -0.4150018095970154, -0.4166463315486908, 0.08268959820270538, -0.5636165142059326, -0.07864285260438919, -0.4870229959487915, 0.853248119354248, -0.08575553447008133, 0.9976483583450317, -0.8775221109390259, 0.5057000517845154, 2.475830554962158, -0.4840763807296753, -0.6525238752365112, -0.8720677495002747, -0.6584398746490479, -1.729222297668457, 1.7211174964904785, 0.29045960307121277, -0.12764254212379456, 0.57439124584198, 1.94387686252594, -1.0474821329116821, 0.21595987677574158, 0.5957233309745789, -0.1815192699432373, 2.6426539421081543, -1.388772964477539, -0.5775231122970581, -0.9683646559715271, 0.25649410486221313, 0.29396599531173706, 1.912894368171692, -1.3025507926940918, 1.0231101512908936, 0.5030227303504944, 0.895500659942627, -0.3867453336715698, -0.40595579147338867, 0.7807344794273376, -0.9711957573890686, -0.6806905269622803, 0.844110906124115, -0.3428886830806732, 1.308658480644226, -0.25199276208877563, 0.2642170786857605, -1.2712424993515015, 0.5134215354919434, -2.673985719680786, -1.2669413089752197, 1.0181928873062134, -0.013286289758980274, 0.7366561889648438, -1.4452241659164429, 0.4786030352115631, 0.4116363823413849, -0.4643518626689911, -0.7404999732971191, 0.4313206076622009, 0.7632013559341431, 0.41705888509750366, -0.08826973289251328, 0.4219769239425659, 0.042643096297979355, -1.5477501153945923, 0.6914786696434021, -0.1386275440454483, -0.1574999839067459], "dtype": "float32"}], "e": 611}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(argmax, json.loads(_TESTS), 'cuda', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")